# Pipeline walkthrough: agentic credit risk (PD → validation)

This notebook walks through every piece of code in the pipeline **in the order it runs**, and
executes each step on a **10% stratified sample** of the public Home Credit data.

**How to read it**

* Each section starts with a short explanation of *what* the stage does and *why*.
* A `show(...)` cell then prints the **actual source code** from the `acr` package. The code
  lives in `src/acr/`, and the notebook displays it rather than copying it, so the two can
  never disagree.
* A run cell (with comments) then executes the stage and shows what it produced.
* Only **aggregates** are displayed (counts, rates, IVs, metrics), never individual loans.

**Core design principle.** LLM agents never fit, tune or alter the model. Everything shown in
this notebook is deterministic: fixed seed, versioned config, the same inputs give the same
numbers. The LLM critic and documentation agents (Phase 6) sit *around* this core.

**Status.** All phases are built (the LLM nodes run in dry-run mode). This walkthrough covers
the deterministic core; Phases 1–5 are built: skeleton, data quality, split, legs, feature selection,
modelling (plus offline tuning), independent validation and the illustrative RBI ECL layer.
The LLM critic, documentation and the grounding check are still stubs; they come in Phase 6.

| § | Stage | Graph node(s) | Main source files |
|---|---|---|---|
| 1 | Configuration | (input to setup) | `config.py`, `config/*.yaml` |
| 2 | State and node contract | (all nodes) | `state.py`, `audit.py` |
| 3 | Graph wiring | (orchestration) | `graph.py` |
| 4 | Run setup | `setup` | `nodes/setup.py` |
| 5 | Ingestion and data quality | `ingest_dq` | `core/io.py`, `core/dq.py` |
| 6 | Split | `split` | `core/split.py` |
| 7 | Four leg agents | `leg_*` | `core/legs/*.py` |
| 8 | Feature selection | `select` | `core/binning.py`, `core/selection.py` |
| 9 | Modelling | `model` | `core/scorecard.py`, `core/challenger.py`, `core/calibration.py` |
| 10 | Independent validation | `validate` | `core/validation.py`, `core/findings.py` |
| 11 | Illustrative RBI ECL | `ecl` | `core/ecl/*.py`, `nodes/ecl.py` |
| 11b | Gates and the rest of the graph | routing, `halt` | `graph.py`, `nodes/halt.py` |
| 12 | Offline tuning | `acr tune` (not in the graph) | `tuning.py` |
| 13 | Tests | | `tests/` |

## 0. Setup

Imports, a `show()` helper to display source code, and a `step()` helper to run one graph node and merge its output into the running state.

In [1]:
# --- Imports --------------------------------------------------------------------------
import inspect
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Code, display

# Make the repo importable when the notebook is opened from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)


def show(obj):
    """Display the real source code of a module, class or function from the acr package."""
    display(Code(inspect.getsource(obj), language="python"))


def show_file(rel_path):
    """Display a non-Python file (e.g. YAML config) from the repo."""
    p = ROOT / rel_path
    display(Code(p.read_text(), language=p.suffix.lstrip(".") or "text"))


def step(node_fn, state):
    """Run one graph node the way LangGraph would, and merge its output into the state.

    Nodes return only the keys they declared in `writes`. The `legs` key uses a merge
    reducer in the real graph, which we mimic here with a dict update.
    """
    out = node_fn(state)
    for k, v in out.items():
        if k in ("legs",):
            state[k] = {**state.get(k, {}), **v}
        elif k in ("gate_log", "llm_calls", "critic_reviews", "extra_analyses"):
            state[k] = state.get(k, []) + v
        else:
            state[k] = v
    print(f"{node_fn.spec.name}: wrote {sorted(out)}")
    return out

## 1. Configuration

All settings live in versioned YAML files under `config/`, validated by **pydantic** models
in `config.py`. Unknown keys are rejected, and limits are enforced in the schema itself
(for example at most **15** scorecard variables and LightGBM `max_depth` between 1 and 10).
The merged config is hashed, and the hash goes into the `run_id`, so every output can be
traced to the exact settings that produced it.

Files: `base.yaml` (data, split, features, model), `gates.yaml` (gate and finding
thresholds), `llm.yaml` (LLM provider, default dry-run), `ecl_rbi.yaml` (RBI ECL parameters
with paragraph citations), `tuning.yaml` (offline tuning grids).

In [2]:
# The modelling and data configuration
show_file("config/base.yaml")

# Versioned run configuration. Any change to a value here changes the config hash in the audit trail.
config_version: "0.1.0"
seed: 42

paths:
  raw_dir: data/raw
  interim_dir: data/interim      # downcast parquet cache built from raw CSVs (gitignored)
  outputs_dir: outputs

runtime:
  # Max nodes LangGraph runs at once. The four legs run in parallel; on an 8 GB machine,
  # 2 keeps peak memory safe on full runs.
  max_concurrency: 2

data:
  id_col: SK_ID_CURR
  target_col: TARGET
  tables:
    application: application_train.csv
    # Kaggle scoring set: no TARGET. Used only for an external score-PSI check and a Kaggle
    # submission export. Optional: if the file is missing, those steps are skipped.
    application_test: application_test.csv
    bureau: bureau.csv
    bureau_balance: bureau_balance.csv
    installments: installments_payments.csv
    pos_cash: POS_CASH_balance.csv
    credit_card: credit_card_balance.csv
    previous: previous_application.csv

dq:
  min_rows: 1000
  target_rate_band: [0.05, 0.12]   # Home Credit's known rate is ~8.1%; outside the band is flagged high
  high_missing_share: 0.50         # columns above this missing share are listed
  duplicate_share_max: 0.001       # child-table duplicate rows above this share are medium severity

split:
  # Step 1: full population -> (dev, oot) 75:25, stratified on TARGET.
  oot_frac: 0.25
  # Step 2: dev -> (train, test) 75:25, stratified on TARGET.
  test_frac_of_dev: 0.25

sample:
  # Used only when --sample is passed. Stratified on TARGET at customer level, before splitting.
  default_frac: 0.10

features:
  # Fitted on train only. Test is used only for the CSI stability screen (D8).
  min_bin_size: 0.05         # each WoE bin holds at least 5% of train
  max_n_bins: 6              # tuned (D22)
  exclude_features: []       # removed from scorecard and challenger candidates (sensitivity runs)
  missing_max: 0.95          # exclude candidates missing for more than 95% of train
  max_features: 15           # scorecard size cap after pruning, ranked by IV (hard limit 15; tuned, D22)
  p_value_max: 0.05          # backward elimination on the WoE logistic regression
  iv_min: 0.02
  iv_suspicious: 0.50        # IV above this is flagged for possible leakage
  corr_max: 0.70
  vif_max: 5.0
  csi_max: 0.10              # train vs test characteristic stability

model:
  # Which model holds the champion role (gates, ECL, documents, reason codes). The other is the
  # benchmark challenger and documented fallback. Decision D33: monotone LightGBM.
  champion: lightgbm         # lightgbm | scorecard
  scorecard:
    pdo: 20
    base_score: 600
    base_odds: 50            # good:bad odds at base_score
  lightgbm_fit:
    validation_frac: 0.10    # internal early-stopping split carved out of train (never test)
    early_stopping_rounds: 50
    # Monotonic constraints: each numeric feature's effect on PD may only move in the direction
    # of its train-only WoE trend (categoricals and flat features stay unconstrained).
    monotone: true
    monotone_method: intermediate   # LightGBM: basic | intermediate | advanced
  calibration:
    cv_folds: 5              # out-of-fold predictions on train drive the calibration decision
    ece_max: 0.005           # recalibrate if OOF decile ECE exceeds this
    slope_tol: 0.10          # or if the OOF calibration slope differs from 1 by more than this
  lightgbm:
    # Tuned by train-only 3-fold CV, 1-SE rule (D22). Hard limit: max_depth 1..10.
    max_depth: 5
    n_estimators: 1000
    learning_rate: 0.03
    num_leaves: 15
    min_child_samples: 1000
    subsample: 0.8
    subsample_freq: 1
    colsample_bytree: 0.7
    reg_lambda: 10.0
  master_scale:
    # Upper PD bounds of grades 1..10; grade D (default) is assigned outside the model.
    pd_upper_bounds: [0.005, 0.01, 0.02, 0.035, 0.05, 0.075, 0.10, 0.15, 0.25, 1.0]

explain:
  # Exact TreeSHAP (LightGBM pred_contrib) for the challenger on every OOT applicant.
  reason_

In [3]:
# Gates (what halts a run) and the validator's finding thresholds
show_file("config/gates.yaml")

# Validation gates. A "red" outcome routes to the halt node.
full:
  enforce: true
  gini_min_test: 0.45
  gini_min_oot: 0.45
  psi_red: 0.25
  psi_amber: 0.10
  dq_halt_on: [critical]
sample:
  # --sample runs are noisier; gates are reported but do not halt.
  enforce: false
  gini_min_test: 0.35
  gini_min_oot: 0.35
  psi_red: 0.25
  psi_amber: 0.10
  dq_halt_on: [critical]

# Thresholds for the independent validator's findings (all runs). Severity scale:
# info < low < medium < high < red. Only the gates above can halt a run.
validation:
  gini_amber_margin: 0.05      # holdout Gini within this of the gate minimum -> medium finding
  csi_amber: 0.10              # per selected feature, train vs test/OOT
  csi_red: 0.25
  binomial_alpha_red: 0.01     # grade-level one-sided binomial test (PD under-prediction)
  binomial_alpha_amber: 0.05
  calibration_ratio_tol: 0.10  # |mean PD / default rate - 1| per segment
  overfit_gap_medium: 0.10     # train Gini minus OOT Gini
  challenger_gap_medium: 0.05  # challenger OOT Gini minus champion OOT Gini
  segment_gini_min: 0.30
  segment_min_n: 1000          # smaller segments are reported but not judged
  external_score_share_max: 0.50   # share of scorecard points range from EXT_SOURCE features
  grade_share_min: 0.01
  grade_share_max: 0.30
  shap_top_overlap_min: 0.70  # top-10 SHAP drivers shared between samples (halves, test, train)
  shap_spearman_min: 0.80      # rank correlation of mean |SHAP| across all features
  bootstrap_n: 200             # resamples for the champion-vs-challenger Gini difference
  stress_young_weights: [3, 10, 50]   # stress-PSI ladder: weight on applicants under 30 in the OOT book

In [4]:
# The pydantic schema that validates every YAML file
import acr.config
show(acr.config)

"""Typed, versioned configuration.

All YAML files under ``config/`` are validated into pydantic models. The canonical JSON
form of the merged config is hashed so every run records exactly which settings produced
its numbers.
"""

from __future__ import annotations

import hashlib
import json
from pathlib import Path
from typing import Literal

import yaml
from pydantic import BaseModel, ConfigDict, Field, model_validator

REPO_ROOT = Path(__file__).resolve().parents[2]
CONFIG_DIR = REPO_ROOT / "config"


# Author's hard limits for a lean model. Neither pipeline runs nor tuning may exceed them.
MAX_SCORECARD_FEATURES = 15
MAX_TREE_DEPTH = 10


def check_depth(depth: int) -> int:
    if not 1 <= int(depth) <= MAX_TREE_DEPTH:
        raise ValueError(f"LightGBM max_depth must be 1..{MAX_TREE_DEPTH} (no unlimited depth), "
                         f"got {depth}")  # fmt: skip
    return int(depth)


class _Strict(BaseModel):
    model_config = ConfigDict(extra="forbid", frozen=True)


class Paths(_Strict):
    raw_dir: str
    interim_dir: str
    outputs_dir: str


class RuntimeCfg(_Strict):
    max_concurrency: int = Field(ge=1)


class DQCfg(_Strict):
    min_rows: int
    target_rate_band: tuple[float, float]
    high_missing_share: float
    duplicate_share_max: float


class DataCfg(_Strict):
    id_col: str
    target_col: str
    tables: dict[str, str]


class SplitCfg(_Strict):
    oot_frac: float = Field(gt=0, lt=1)
    test_frac_of_dev: float = Field(gt=0, lt=1)


class SampleCfg(_Strict):
    default_frac: float = Field(gt=0, le=1)


class FeatureCfg(_Strict):
    min_bin_size: float
    max_n_bins: int
    missing_max: float
    max_features: int = Field(ge=1, le=MAX_SCORECARD_FEATURES)
    p_value_max: float
    iv_min: float
    iv_suspicious: float
    corr_max: float
    vif_max: float
    csi_max: float
    exclude_features: list[str] = []


class ScorecardCfg(_Strict):
    pdo: float
    base_score: float
    base_odds: float


class MasterScaleCfg(_Strict):
    pd_upper_bounds: list[float]

    @model_validator(mode="after")
    def _monotone(self) -> MasterScaleCfg:
        b = self.pd_upper_bounds
        if any(x >= y for x, y in zip(b, b[1:], strict=False)) or b[-1] != 1.0:
            raise ValueError("master scale bounds must be strictly increasing and end at 1.0")
        return self


class LGBMFitCfg(_Strict):
    validation_frac: float
    early_stopping_rounds: int
    monotone: bool = False
    monotone_method: Literal["basic", "intermediate", "advanced"] = "intermediate"


class ReasonFamily(_Strict):
    family: str
    label: str
    patterns: list[str]


class ExplainCfg(_Strict):
    reason_codes: int = Field(ge=1, le=10)
    stability_top_n: int = Field(ge=3, le=30)
    families: list[ReasonFamily] = []


class CalibrationCfg(_Strict):
    cv_folds: int
    ece_max: float
    slope_tol: float


class ModelCfg(_Strict):
    champion: Literal["lightgbm", "scorecard"] = "lightgbm"
    scorecard: ScorecardCfg
    lightgbm_fit: LGBMFitCfg
    calibration: CalibrationCfg
    lightgbm: dict[str, float | int]

    @model_validator(mode="after")
    def _lean_trees(self) -> ModelCfg:
        if "max_depth" not in self.lightgbm:
            raise ValueError("LightGBM max_depth must be set explicitly (limit 10)")
        check_depth(self.lightgbm["max_depth"])
        return self

    master_scale: MasterScaleCfg


class BaseCfg(_Strict):
    config_version: str
    seed: int
    paths: Paths
    runtime: RuntimeCfg
    data: DataCfg
    dq: DQCfg
    split: SplitCfg
    sample: SampleCfg
    features: FeatureCfg
    model: ModelCfg
    explain: ExplainCfg


class GateSet(_Strict):
    enforce: bool
    gini_min_test: float
    gini_min_oot: float
    psi_red: float
    psi_amber: float
    dq_halt_on: list[str]


class ValidationCfg(_Strict):
    gini_amber_margin: float
    csi_amber: float
    csi_red: float
    binomial_alpha_red: float
    binomial_alpha_amber: float
    calibration_ratio_tol: floa

In [5]:
# Load the config exactly as the CLI does. The default LLM mode is dry-run (no API calls).
from acr.config import load_config

cfg = load_config(llm_mode="dry-run")
print("config hash:", cfg.hash())
print("seed:", cfg.base.seed)
print("scorecard feature cap:", cfg.base.features.max_features)
print("LightGBM params:", cfg.base.model.lightgbm)

config hash: bd0bb86ae168
seed: 42
scorecard feature cap: 15
LightGBM params: {'max_depth': 5, 'n_estimators': 1000, 'learning_rate': 0.03, 'num_leaves': 15, 'min_child_samples': 1000, 'subsample': 0.8, 'subsample_freq': 1, 'colsample_bytree': 0.7, 'reg_lambda': 10.0}


## 2. Typed state and the node contract

**State.** `PipelineState` is the single typed object that flows through the LangGraph
graph. It carries *artefact paths, metrics and findings*, never DataFrames. Tables are saved
as parquet under `outputs/<run_id>/artefacts/`. This keeps the state auditable, and it means
anything later handed to an LLM is a summary by construction.

**Node contract.** Every node is wrapped by the `@node(...)` decorator in `audit.py`, which:
1. gives the node a **read-only view** holding only the state keys it declared in `reads`;
2. **rejects** any returned key not declared in `writes`;
3. writes start/end/error events, input and output summaries and timings to `audit.jsonl`.

In [6]:
import acr.state
show(acr.state)

"""Typed pipeline state.

Design rule: state carries artefact *paths*, metric dictionaries, findings and audit metadata.
It never carries DataFrames. Tables live as parquet under ``outputs/<run_id>/artefacts/``.
That keeps state serialisable and auditable, and it means anything handed to an LLM is, by
construction, a summary rather than loan-level data.
"""

from __future__ import annotations

import operator
from typing import Annotated, Any, Literal, TypedDict

from pydantic import BaseModel, ConfigDict

Severity = Literal["info", "low", "medium", "high", "red"]


def merge_dicts(left: dict | None, right: dict | None) -> dict:
    """Reducer for keys written by parallel branches (the leg fan-out)."""
    out = dict(left or {})
    for k, v in (right or {}).items():
        if k in out:
            raise ValueError(f"parallel branches both wrote '{k}'")
        out[k] = v
    return out


class Finding(BaseModel):
    """A structured validation or critic finding."""

    model_config = ConfigDict(extra="forbid", frozen=True)

    id: str
    area: str
    severity: Severity
    title: str
    detail: str
    remediation: str = ""


# Whitelisted deterministic analyses the critic may request. The critic can only pick from
# this menu; there is no request type that touches model parameters.
AnalysisKind = Literal[
    "segment_cut",
    "lgd_sensitivity",
    "binning_stability_alt",
    "grade_concentration",
    "pd_floor_impact",
]


class CriticRequest(BaseModel):
    model_config = ConfigDict(extra="forbid", frozen=True)

    kind: AnalysisKind
    params: dict[str, str | int | float] = {}
    rationale: str


class LLMCall(BaseModel):
    model_config = ConfigDict(extra="forbid", frozen=True)

    node: str
    provider: str
    model: str
    input_tokens: int
    output_tokens: int
    cost_usd: float


class PipelineState(TypedDict, total=False):
    # --- run setup (orchestrator) ---
    run_id: str
    run_dir: str
    config: dict[str, Any]  # validated AppConfig dumped to JSON-safe dict
    config_hash: str
    sample_frac: float | None  # None = full population
    llm_mode: str

    # --- ingestion and data quality ---
    dq: dict[str, Any]  # checks, severities, target rate, table shapes

    # --- split ---
    split: dict[str, Any]  # id-list paths and proportions per partition

    # --- legs (parallel fan-out; merged by reducer) ---
    legs: Annotated[dict[str, Any], merge_dicts]

    # --- feature assembly and selection ---
    selection: dict[str, Any]

    # --- modelling (champion + challenger) ---
    model: dict[str, Any]

    # --- explainability (SHAP for the challenger) ---
    explain: dict[str, Any]

    # --- independent validation (read-only over model artefacts) ---
    validation: dict[str, Any]

    # --- ECL ---
    ecl: dict[str, Any]

    # --- critic loop ---
    critic_iteration: int
    critic_requests: list[dict[str, Any]]  # pending, for this iteration
    critic_reviews: Annotated[list[dict[str, Any]], operator.add]
    extra_analyses: Annotated[list[dict[str, Any]], operator.add]

    # --- documentation and grounding ---
    documents: dict[str, str]  # doc name -> markdown path
    grounding: dict[str, Any]

    # --- cross-cutting ---
    llm_calls: Annotated[list[dict[str, Any]], operator.add]
    gate_log: Annotated[list[dict[str, Any]], operator.add]
    halted: bool
    halt_reason: str
    report: dict[str, Any]

In [7]:
import acr.audit
show(acr.audit)

"""Append-only JSONL audit trail and the node contract.

Every graph node is wrapped by :func:`node`, which:

* gives the node a read-only view containing only the state keys it declared in ``reads``
  (reading anything else raises ``KeyError``);
* rejects any returned key not declared in ``writes``;
* logs start, end (or error), input and output summaries, and timing to ``audit.jsonl``.

The declarations are also kept in a registry, which the README generator uses to build the
per-node table, so the documentation cannot drift from the code.
"""

from __future__ import annotations

import json
import time
from collections.abc import Callable, Mapping
from dataclasses import dataclass
from datetime import UTC, datetime
from pathlib import Path
from types import MappingProxyType
from typing import Any

from acr.state import PipelineState

AUDIT_FILE = "audit.jsonl"


class ContractViolation(RuntimeError):
    """A node read or wrote a state key outside its declared contract."""


def _now() -> str:
    return datetime.now(UTC).isoformat(timespec="milliseconds")


def _summarise(value: Any, depth: int = 0) -> Any:
    """Compact, JSON-safe summary of a state value for the audit trail."""
    if isinstance(value, (int, float, bool)) or value is None:
        return value
    if isinstance(value, str):
        return value if len(value) <= 120 else value[:117] + "..."
    if isinstance(value, Mapping):
        if depth >= 1:
            return {"type": "dict", "keys": sorted(map(str, value))[:12]}
        return {str(k): _summarise(v, depth + 1) for k, v in list(value.items())[:20]}
    if isinstance(value, (list, tuple)):
        return {"type": "list", "len": len(value)}
    return {"type": type(value).__name__}


def write_event(run_dir: str | Path, event: dict[str, Any]) -> None:
    path = Path(run_dir) / AUDIT_FILE
    record = {"ts": _now(), **event}
    with open(path, "a") as f:
        f.write(json.dumps(record, default=str) + "\n")


def read_events(run_dir: str | Path) -> list[dict[str, Any]]:
    path = Path(run_dir) / AUDIT_FILE
    if not path.exists():
        return []
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


@dataclass(frozen=True)
class NodeSpec:
    name: str
    role: str
    reads: tuple[str, ...]
    writes: tuple[str, ...]
    uses_llm: bool
    on_failure: str


NODE_REGISTRY: dict[str, NodeSpec] = {}


def node(
    name: str,
    *,
    role: str,
    reads: list[str],
    writes: list[str],
    uses_llm: bool = False,
    on_failure: str = "Exception propagates; run stops and the audit trail records the error.",
) -> Callable[[Callable[[Mapping[str, Any]], dict[str, Any]]], Callable[[PipelineState], dict]]:
    """Decorator enforcing a node's declared read/write contract and auditing it."""
    always_readable = ("run_id", "run_dir")
    allowed_reads = tuple(dict.fromkeys([*always_readable, *reads]))
    allowed_writes = tuple(writes)

    def deco(fn: Callable[[Mapping[str, Any]], dict[str, Any]]) -> Callable[[PipelineState], dict]:
        spec = NodeSpec(name, role, tuple(reads), allowed_writes, uses_llm, on_failure)
        NODE_REGISTRY[name] = spec

        def wrapped(state: PipelineState) -> dict[str, Any]:
            run_dir = state.get("run_dir")
            view = MappingProxyType({k: state[k] for k in allowed_reads if k in state})
            if run_dir:
                write_event(
                    run_dir,
                    {
                        "event": "node_start",
                        "node": name,
                        "inputs": {k: _summarise(view[k]) for k in reads if k in view},
                    },
                )
            t0 = time.perf_counter()
            try:
                out = fn(view) or {}
            except Exception as exc:
                if run_dir:
                    write_event(
                        run_dir,
                        {
                            "event": "node_error",
    

## 3. Graph wiring

`graph.py` assembles the LangGraph `StateGraph`:

* **Conditional edges** implement the gates: after data quality (`ingest_dq` → `split` or
  `halt`) and after validation (`validate` → `ecl` or `halt`).
* **Fan-out and join**: `split` fans out to the four leg nodes, which run in parallel, and
  `select` waits for all four.
* **Bounded critic loop**: `critic` → `run_extra_analyses` → `critic`, at most two rounds.

The Mermaid diagram below is generated from the compiled graph, not drawn by hand.

In [8]:
import acr.graph
show(acr.graph)

"""LangGraph wiring: nodes, gates (conditional edges), leg fan-out/join and the critic loop."""

from __future__ import annotations

from langgraph.graph import END, START, StateGraph

from acr.audit import write_event
from acr.config import AppConfig
from acr.nodes.critic import critic
from acr.nodes.document import document
from acr.nodes.ecl import ecl
from acr.nodes.explain import explain
from acr.nodes.extras import run_extra_analyses
from acr.nodes.grounding import grounding_check
from acr.nodes.halt import halt
from acr.nodes.ingest_dq import ingest_dq
from acr.nodes.legs import LEG_NODES
from acr.nodes.model import model
from acr.nodes.report import assemble_report
from acr.nodes.select import select
from acr.nodes.setup import setup
from acr.nodes.split import split
from acr.nodes.validate import validate
from acr.state import PipelineState


def _log_route(state: PipelineState, gate: str, decision: str, detail: str = "") -> str:
    write_event(
        state["run_dir"], {"event": "route", "gate": gate, "to": decision, "detail": detail}
    )
    return decision


def route_after_dq(state: PipelineState) -> str:
    status = state.get("dq", {}).get("status")
    return _log_route(state, "data_quality", "halt" if status == "halt" else "split", str(status))


def route_after_validation(state: PipelineState) -> str:
    status = state.get("validation", {}).get("status")
    return _log_route(state, "validation", "halt" if status == "halt" else "ecl", str(status))


def route_after_critic(state: PipelineState) -> str:
    """Bounded critic loop.

    ``critic_iteration`` counts completed critic reviews. Extra analyses run only while the
    number of completed request rounds is below ``max_critic_iterations``; after that, any
    new requests are dropped and logged, and the run moves on to documentation.
    """
    max_rounds = AppConfig.model_validate(state["config"]).llm.max_critic_iterations
    requests = state.get("critic_requests") or []
    rounds_done = state.get("critic_iteration", 0) - 1  # extra-analysis rounds already run
    if requests and rounds_done < max_rounds:
        return _log_route(state, "critic_loop", "run_extra_analyses", f"{len(requests)} request(s)")
    detail = "no requests" if not requests else f"bound reached; {len(requests)} request(s) dropped"
    return _log_route(state, "critic_loop", "document", detail)


def build_graph():
    g = StateGraph(PipelineState)

    g.add_node("setup", setup)
    g.add_node("ingest_dq", ingest_dq)
    g.add_node("split", split)
    for name, fn in LEG_NODES.items():
        g.add_node(name, fn)
    g.add_node("select", select)
    g.add_node("model", model)
    g.add_node("explain", explain)
    g.add_node("validate", validate)
    g.add_node("ecl", ecl)
    g.add_node("critic", critic)
    g.add_node("run_extra_analyses", run_extra_analyses)
    g.add_node("document", document)
    g.add_node("grounding_check", grounding_check)
    g.add_node("assemble_report", assemble_report)
    g.add_node("halt", halt)

    g.add_edge(START, "setup")
    g.add_edge("setup", "ingest_dq")
    g.add_conditional_edges("ingest_dq", route_after_dq, {"split": "split", "halt": "halt"})

    # Fan-out: split -> four legs in parallel. Join: select waits for all four.
    for name in LEG_NODES:
        g.add_edge("split", name)
    g.add_edge(list(LEG_NODES), "select")

    g.add_edge("select", "model")
    g.add_edge("model", "explain")
    g.add_edge("explain", "validate")
    g.add_conditional_edges("validate", route_after_validation, {"ecl": "ecl", "halt": "halt"})
    g.add_edge("ecl", "critic")
    g.add_conditional_edges(
        "critic",
        route_after_critic,
        {"run_extra_analyses": "run_extra_analyses", "document": "document"},
    )
    g.add_edge("run_extra_analyses", "critic")
    g.add_edge("document", "grounding_check")
    g.add_edge("grounding_check", "assemble_report")
    g.add_edge("assemble_report", END)
    g.add_edge("halt", END)

    return

In [9]:
# Mermaid source generated from the compiled LangGraph (dotted arrows = conditional edges)
from acr.graph import build_graph, mermaid

graph = build_graph()
print(mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	setup(setup)
	ingest_dq(ingest_dq)
	split(split)
	leg_application(leg_application)
	leg_bureau(leg_bureau)
	leg_behavioural(leg_behavioural)
	leg_previous(leg_previous)
	select(select)
	model(model)
	explain(explain)
	validate(validate)
	ecl(ecl)
	critic(critic)
	run_extra_analyses(run_extra_analyses)
	document(document)
	grounding_check(grounding_check)
	assemble_report(assemble_report)
	halt(halt)
	__end__([<p>__end__</p>]):::last
	__start__ --> setup;
	critic -.-> document;
	critic -.-> run_extra_analyses;
	document --> grounding_check;
	ecl --> critic;
	explain --> validate;
	grounding_check --> assemble_report;
	ingest_dq -.-> halt;
	ingest_dq -.-> split;
	leg_application --> select;
	leg_behavioural --> select;
	leg_bureau --> select;
	leg_previous --> select;
	model --> explain;
	run_extra_analyses --> critic;
	select --> model;
	setup --> ingest_dq;
	split --> leg_application;
	spli

In [10]:
# Every node's declared contract, straight from the decorator registry
from acr.audit import NODE_REGISTRY

pd.DataFrame([
    {"node": s.name, "uses_llm": s.uses_llm, "reads": ", ".join(s.reads), "writes": ", ".join(s.writes)}
    for s in NODE_REGISTRY.values()
])

,node,uses_llm,reads,writes
0,critic,True,"config, config_hash, sample_frac, dq, split, s...","critic_reviews, critic_requests, critic_iterat..."
1,document,True,"config, config_hash, sample_frac, dq, split, s...","documents, llm_calls"
2,ecl,False,"config, split, dq, selection, model",ecl
3,explain,False,"config, split, selection, model",explain
4,run_extra_analyses,False,"config, critic_requests, critic_iteration, mod...","extra_analyses, critic_requests"
5,grounding_check,False,"config, config_hash, sample_frac, dq, split, s...",grounding
6,assemble_report,False,"run_id, config_hash, sample_frac, llm_mode, dq...",report
7,halt,False,"run_id, config_hash, sample_frac, llm_mode, dq...","halted, halt_reason, report"
8,ingest_dq,False,"config, sample_frac","dq, gate_log"
9,leg_application,False,"config, split, dq",legs


## 4. Run setup (node `setup`)

Validates the config, builds a `run_id` (timestamp, scope, config hash), creates
`outputs/<run_id>/`, snapshots the config and starts the audit trail.

Below we run the nodes **one at a time** on a 10% sample, the same way the graph would,
so we can inspect each step. (`acr run --sample` does all of this in one go.)

In [11]:
import acr.nodes.setup
show(acr.nodes.setup)

"""Node 0: orchestrator / run setup."""

from __future__ import annotations

import json
from datetime import UTC, datetime
from pathlib import Path

from acr.audit import node, write_event
from acr.config import AppConfig, resolve_path


@node(
    "setup",
    role="Validate config, create run_id and run folder, snapshot config, start the audit trail.",
    reads=["config", "sample_frac", "llm_mode"],
    writes=["run_id", "run_dir", "config_hash", "critic_iteration", "critic_requests", "halted"],
    on_failure="Invalid config raises a pydantic ValidationError before any data is touched.",
)
def setup(view):
    cfg = AppConfig.model_validate(view["config"])
    h = cfg.hash()
    frac = view.get("sample_frac")
    scope = "full" if frac is None else f"s{round(frac * 100):02d}"
    run_id = f"{datetime.now(UTC):%Y%m%dT%H%M%SZ}_{scope}_{h}"
    run_dir = resolve_path(cfg.base.paths.outputs_dir) / run_id
    (run_dir / "artefacts").mkdir(parents=True, exist_ok=False)
    (Path(run_dir) / "config_snapshot.json").write_text(
        json.dumps(cfg.model_dump(mode="json"), indent=2, sort_keys=True)
    )
    write_event(
        run_dir,
        {
            "event": "run_start",
            "run_id": run_id,
            "config_hash": h,
            "config_version": cfg.base.config_version,
            "seed": cfg.base.seed,
            "sample_frac": frac,
            "llm_mode": view.get("llm_mode"),
        },
    )
    return {
        "run_id": run_id,
        "run_dir": str(run_dir),
        "config_hash": h,
        "critic_iteration": 0,
        "critic_requests": [],
        "halted": False,
    }

In [12]:
from acr.nodes.setup import setup

# Initial input to the graph: validated config, sample fraction, LLM mode
state = {"config": cfg.model_dump(mode="json"), "sample_frac": 0.10, "llm_mode": "dry-run"}
step(setup, state)
print("run_id:", state["run_id"])

setup: wrote ['config_hash', 'critic_iteration', 'critic_requests', 'halted', 'run_dir', 'run_id']
run_id: 20261019T060128Z_s10_bd0bb86ae168


## 5. Ingestion and data quality (node `ingest_dq`)

**Memory-conscious loading (`core/io.py`).** The raw CSVs (2.5 GB) are converted once into a
downcast, zstd-compressed parquet cache (about 513 MB): floats to float32 (amounts used in
ECL stay float64), integer-valued columns to the smallest integer type, strings to
dictionary-encoded categoricals. Later reads select only the needed columns and filter rows
by ID at read time.

**Data-quality checks (`core/dq.py`).** Each check has a severity that applies if it fails.
A failed **critical** check (missing table or column, duplicate IDs, non-binary target)
sets `status = "halt"`, and the graph's conditional edge routes to the `halt` node.
DQ always runs on the full population, even on a sample run.

In [13]:
import acr.core.schema
show(acr.core.schema)

"""Columns each table must provide. Data-quality checks fail critically if any are missing."""

from __future__ import annotations

ID = "SK_ID_CURR"
TARGET = "TARGET"

# Home Credit encodes "not applicable / far future" in DAYS_* columns as 365243.
DAYS_SENTINEL = 365243

APP_NUMERIC = [
    "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3",
    "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE",
    "DAYS_BIRTH", "DAYS_EMPLOYED", "DAYS_REGISTRATION", "DAYS_ID_PUBLISH",
    "DAYS_LAST_PHONE_CHANGE", "CNT_CHILDREN", "CNT_FAM_MEMBERS",
    "REGION_POPULATION_RELATIVE", "REGION_RATING_CLIENT_W_CITY", "OWN_CAR_AGE",
    "AMT_REQ_CREDIT_BUREAU_MON", "AMT_REQ_CREDIT_BUREAU_QRT", "AMT_REQ_CREDIT_BUREAU_YEAR",
    "DEF_30_CNT_SOCIAL_CIRCLE", "DEF_60_CNT_SOCIAL_CIRCLE", "OBS_30_CNT_SOCIAL_CIRCLE",
    "FLAG_DOCUMENT_3", "REG_CITY_NOT_WORK_CITY", "REG_CITY_NOT_LIVE_CITY",
    "FLAG_WORK_PHONE", "FLAG_PHONE", "TOTALAREA_MODE",
]  # fmt: skip
APP_CATEGORICAL = [
    "NAME_CONTRACT_TYPE", "FLAG_OWN_CAR", "FLAG_OWN_REALTY", "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE", "NAME_HOUSING_TYPE", "OCCUPATION_TYPE", "ORGANIZATION_TYPE",
]  # fmt: skip
# Protected characteristics: kept for segment and fairness analysis, never offered to the model.
APP_PROTECTED = ["CODE_GENDER", "NAME_FAMILY_STATUS"]

REQUIRED_COLUMNS: dict[str, list[str]] = {
    "application": [ID, TARGET, *APP_NUMERIC, *APP_CATEGORICAL, *APP_PROTECTED],
    "application_test": [ID, *APP_NUMERIC, *APP_CATEGORICAL, *APP_PROTECTED],
    "bureau": [
        ID, "SK_ID_BUREAU", "CREDIT_ACTIVE", "DAYS_CREDIT", "CREDIT_DAY_OVERDUE",
        "DAYS_CREDIT_ENDDATE", "AMT_CREDIT_MAX_OVERDUE", "CNT_CREDIT_PROLONG", "AMT_CREDIT_SUM",
        "AMT_CREDIT_SUM_DEBT", "AMT_CREDIT_SUM_OVERDUE", "CREDIT_TYPE",
    ],
    "bureau_balance": ["SK_ID_BUREAU", "MONTHS_BALANCE", "STATUS"],
    "installments": [
        "SK_ID_PREV", ID, "NUM_INSTALMENT_VERSION", "NUM_INSTALMENT_NUMBER", "DAYS_INSTALMENT",
        "DAYS_ENTRY_PAYMENT", "AMT_INSTALMENT", "AMT_PAYMENT",
    ],
    "pos_cash": [
        "SK_ID_PREV", ID, "MONTHS_BALANCE", "CNT_INSTALMENT_FUTURE", "NAME_CONTRACT_STATUS",
        "SK_DPD_DEF",
    ],
    "credit_card": [
        "SK_ID_PREV", ID, "MONTHS_BALANCE", "AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL",
        "AMT_DRAWINGS_ATM_CURRENT", "AMT_PAYMENT_CURRENT", "AMT_INST_MIN_REGULARITY", "SK_DPD_DEF",
    ],
    "previous": [
        "SK_ID_PREV", ID, "NAME_CONTRACT_TYPE", "AMT_ANNUITY", "AMT_APPLICATION", "AMT_CREDIT",
        "RATE_DOWN_PAYMENT", "NAME_CONTRACT_STATUS", "DAYS_DECISION", "CNT_PAYMENT",
        "NAME_YIELD_GROUP", "DAYS_TERMINATION",
    ],
}  # fmt: skip

# Amounts kept at float64 in the cache (they feed EIR and EAD maths in the ECL layer).
_AMOUNTS = {"AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE"}
KEEP_FLOAT64 = {"application": _AMOUNTS, "application_test": _AMOUNTS}

# Tables whose absence is not critical: the dependent steps are skipped instead.
OPTIONAL_TABLES = {"application_test"}

BUREAU_STATUS_DOMAIN = {"0", "1", "2", "3", "4", "5", "C", "X"}

In [14]:
import acr.core.io
show(acr.core.io)

"""Memory-conscious data access.

Raw CSVs are converted once into a downcast parquet cache (``data/interim``):

* floats go to float32, except amounts listed in ``schema.KEEP_FLOAT64``;
* integer-valued columns with no nulls go to the smallest integer type that fits;
* strings are dictionary-encoded (they read back as pandas categoricals).

The cache is rebuilt when the source file's size or modification time changes. Downstream
code reads only the columns it needs and filters rows by ID at read time.
"""

from __future__ import annotations

import json
from collections.abc import Iterable
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.csv as pacsv
import pyarrow.parquet as pq

from acr.core.schema import KEEP_FLOAT64

CACHE_VERSION = 1
MANIFEST = "manifest.json"


def _infer_types(path: Path, n: int = 200_000) -> dict[str, pa.DataType]:
    """Fix column types from a large head sample so later CSV blocks cannot disagree."""
    head = pd.read_csv(path, nrows=n)
    types: dict[str, pa.DataType] = {}
    for c in head.columns:
        if c.startswith("SK_ID") or c == "TARGET":
            types[c] = pa.int64()
        elif pd.api.types.is_numeric_dtype(head[c]):
            types[c] = pa.float64()
        else:
            types[c] = pa.string()
    return types


def _int_type(lo: int, hi: int) -> pa.DataType:
    for t, info in ((pa.int8(), np.iinfo(np.int8)), (pa.int16(), np.iinfo(np.int16)),
                    (pa.int32(), np.iinfo(np.int32))):  # fmt: skip
        if lo >= info.min and hi <= info.max:
            return t
    return pa.int64()


def downcast(tbl: pa.Table, keep_float64: Iterable[str] = ()) -> pa.Table:
    keep = set(keep_float64)
    cols = []
    for name, col in zip(tbl.column_names, tbl.columns, strict=True):
        t = col.type
        if pa.types.is_integer(t) or (pa.types.is_floating(t) and name not in keep):
            mm = pc.min_max(col).as_py()
            integral = (
                col.null_count == 0
                and mm["min"] is not None
                and (pa.types.is_integer(t) or pc.all(pc.equal(col, pc.floor(col))).as_py())
            )
            if integral:
                col = col.cast(_int_type(int(mm["min"]), int(mm["max"])))
            elif pa.types.is_floating(t):
                col = col.cast(pa.float32())
        elif pa.types.is_string(t) or pa.types.is_large_string(t):
            col = pc.dictionary_encode(col)
        cols.append(col)
    return pa.table(cols, names=tbl.column_names)


def _fingerprint(path: Path) -> dict:
    st = path.stat()
    return {"size": st.st_size, "mtime_ns": st.st_mtime_ns, "cache_version": CACHE_VERSION}


def ensure_parquet(raw_dir: Path, interim_dir: Path, tables: dict[str, str]) -> dict[str, dict]:
    """Build or reuse the parquet cache. Returns {table: {path, rows, cols, rebuilt}}."""
    interim_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = interim_dir / MANIFEST
    manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
    out: dict[str, dict] = {}
    for name, fname in tables.items():
        src = raw_dir / fname
        if not src.exists():
            out[name] = {"path": None, "missing": str(src)}
            continue
        dst = interim_dir / f"{name}.parquet"
        fp = _fingerprint(src)
        rebuilt = False
        if not dst.exists() or manifest.get(name) != fp:
            tbl = pacsv.read_csv(
                src, convert_options=pacsv.ConvertOptions(column_types=_infer_types(src))
            )
            tbl = downcast(tbl, KEEP_FLOAT64.get(name, ()))
            pq.write_table(tbl, dst, compression="zstd")
            del tbl
            manifest[name] = fp
            manifest_path.write_text(json.dumps(manifest, indent=2))
            rebuilt = True
        meta = pq.ParquetFile(dst).metadata
        out[name] = {"path": str(dst), "rows": meta.num_rows, "cols": meta.num_colum

In [15]:
import acr.core.dq
show(acr.core.dq)

"""Data-quality checks. Each check reports a severity that applies when it fails.

Severities: critical (halts the run), high, medium, low, info.
"""

from __future__ import annotations

from dataclasses import asdict, dataclass, field
from typing import Any

import pandas as pd
import pyarrow.parquet as pq

from acr.config import DQCfg
from acr.core.io import column_null_counts, read
from acr.core.schema import (
    BUREAU_STATUS_DOMAIN,
    DAYS_SENTINEL,
    ID,
    OPTIONAL_TABLES,
    REQUIRED_COLUMNS,
    TARGET,
)

SEVERITY_ORDER = ["info", "low", "medium", "high", "critical"]


@dataclass
class Check:
    name: str
    table: str
    severity: str
    passed: bool
    detail: str
    value: Any = None
    extra: dict = field(default_factory=dict)


def run_checks(tables: dict[str, dict], cfg: DQCfg) -> list[Check]:
    checks: list[Check] = []

    for name, info in tables.items():
        present = info.get("path") is not None
        sev = "medium" if name in OPTIONAL_TABLES else "critical"
        checks.append(Check("table_present", name, sev, present,
                            "loaded" if present else f"missing file {info.get('missing')}"))  # fmt: skip
    present = {n: i["path"] for n, i in tables.items() if i.get("path")}

    for name, path in present.items():
        cols = set(pq.ParquetFile(path).schema_arrow.names)
        missing = [c for c in REQUIRED_COLUMNS.get(name, []) if c not in cols]
        checks.append(Check("required_columns", name, "critical", not missing,
                            f"missing: {missing}" if missing else "all required columns present",
                            value=len(missing)))  # fmt: skip
    if any(c.name == "required_columns" and not c.passed for c in checks):
        return checks  # downstream checks would fail on missing columns

    if "application" in present:
        checks += _application_checks(present["application"], cfg)
        app_ids = set(read(present["application"], [ID])[ID])
    else:
        app_ids = set()

    for name, path in present.items():
        rows = pq.ParquetFile(path).metadata.num_rows
        nulls = column_null_counts(path)
        high = sorted(c for c, n in nulls.items() if rows and n / rows > cfg.high_missing_share)
        checks.append(Check("high_missing_columns", name, "low", not high,
                            f"{len(high)} column(s) over {cfg.high_missing_share:.0%} missing",
                            value=len(high), extra={"columns": high}))  # fmt: skip
        if name == "application":
            continue
        if name == "application_test":
            ext = read(path, [ID])[ID]
            checks.append(Check("id_unique", name, "high", ext.is_unique,
                                f"{int(ext.duplicated().sum())} duplicate {ID}"))  # fmt: skip
            overlap = int(ext.isin(app_ids).sum())
            checks.append(Check("disjoint_from_train", name, "high", overlap == 0,
                                f"{overlap} {ID} also appear in application_train"))  # fmt: skip
            checks.append(Check("unlabelled", name, "info", True,
                                f"{len(ext)} applicants with no TARGET: used for an external "
                                "score-PSI check and a Kaggle submission only"))  # fmt: skip
            continue
        df = read(path, REQUIRED_COLUMNS[name])
        dup = int(df.duplicated().sum())
        share = dup / max(len(df), 1)
        checks.append(Check("duplicate_rows", name, "medium", share <= cfg.duplicate_share_max,
                            f"{dup} duplicate rows ({share:.4%})", value=dup))  # fmt: skip
        if ID in df.columns:
            null_ids = int(df[ID].isna().sum())
            checks.append(Check("key_not_null", name, "high", null_ids == 0,
                                f"{null_ids} null {ID}", value=null_ids))  # fmt: skip
            orphan = 1 - df[ID].isin(app_ids).mean() if app_ids else 0.0
            checks.append(Check("orphan_key_share"

In [16]:
import acr.nodes.ingest_dq
show(acr.nodes.ingest_dq)

"""Node 1: ingestion and data quality."""

from __future__ import annotations

from acr.audit import node
from acr.config import resolve_path
from acr.core.dq import run_checks, summarise
from acr.nodes._util import cfg, is_sample


@node(
    "ingest_dq",
    role="Load raw tables into a downcast parquet cache; run schema, key, duplicate, missingness "
    "and target-rate checks.",
    reads=["config", "sample_frac"],
    writes=["dq", "gate_log"],
    on_failure="Any failed critical check sets dq.status='halt'; the gate routes to the halt node.",
)
def ingest_dq(view):
    from acr.core.io import ensure_parquet

    c = cfg(view)
    tables = ensure_parquet(
        resolve_path(c.base.paths.raw_dir),
        resolve_path(c.base.paths.interim_dir),
        c.base.data.tables,
    )
    checks = run_checks(tables, c.base.dq)
    dq = summarise(checks, c.gate_set(is_sample(view)).dq_halt_on)
    dq["tables"] = tables
    rate = next((x["value"] for x in dq["checks"] if x["name"] == "target_rate"), None)
    dq["target_rate"] = rate
    if dq["status"] == "halt":
        gate = {
            "gate": "data_quality",
            "decision": "halt",
            "reason": "; ".join(dq["halting_checks"]),
        }
    else:
        fails = sum(dq["failed_by_severity"].values())
        gate = {"gate": "data_quality", "decision": "continue",
                "reason": f"no critical failures; {fails} non-critical finding(s)"}  # fmt: skip
    return {"dq": dq, "gate_log": [gate]}

In [17]:
from acr.nodes.ingest_dq import ingest_dq

step(ingest_dq, state)

# Tables loaded into the cache
display(pd.DataFrame(state["dq"]["tables"]).T[["rows", "cols"]])

# Every check: failed ones first. Severity applies only when a check fails.
checks = pd.DataFrame(state["dq"]["checks"])[["table", "name", "severity", "passed", "detail"]]
display(checks.sort_values("passed"))
print("DQ status:", state["dq"]["status"], "| target rate:", state["dq"]["target_rate"])

ingest_dq: wrote ['dq', 'gate_log']


,rows,cols
application,307511,122
application_test,48744,121
bureau,1716428,17
bureau_balance,27299925,3
installments,13605401,8
pos_cash,10001358,8
credit_card,3840312,23
previous,1670214,37


,table,name,severity,passed,detail
23,application,high_missing_columns,low,False,38 column(s) over 50% missing
47,previous,high_missing_columns,low,False,4 column(s) over 50% missing
28,bureau,high_missing_columns,low,False,2 column(s) over 50% missing
21,application,annuity_present,medium,False,12 missing AMT_ANNUITY (ECL term and EIR need it)
24,application_test,high_missing_columns,low,False,28 column(s) over 50% missing
0,application,table_present,critical,True,loaded
29,bureau,duplicate_rows,medium,True,0 duplicate rows (0.0000%)
30,bureau,key_not_null,high,True,0 null SK_ID_CURR
31,bureau,orphan_key_share,info,True,14.6% of rows belong to customers outside appl...
32,bureau_balance,high_missing_columns,low,True,0 column(s) over 50% missing


DQ status: pass | target rate: 0.080729


## 6. Split (node `split`)

Exactly as specified: the full population (or the stratified sample) is split **75:25** into
development and OOT, then development is split **75:25** into train and test, both
stratified on `TARGET`, with a fixed seed. Final shares: **56.25 / 18.75 / 25**.

The split happens on `SK_ID_CURR` **before anything learns from the data**. Home Credit has
no application dates, so the OOT is a **pseudo-OOT** (random holdout), not a temporal one.

**`application_test.csv`** is Kaggle's scoring set and has **no TARGET**, so it can't be a
validation partition. It is kept separate as an **external, unlabelled** population. Legs
build its features and the model scores it, but nothing is fitted on it. It feeds a
score-PSI and CSI check in validation, and full runs export Kaggle submission files.

In [18]:
import acr.core.split
show(acr.core.split)

"""Customer-level stratified split: (dev 75 : OOT 25), then dev -> (train 75 : test 25).

Final shares: train 56.25%, test 18.75%, OOT 25%. The OOT is a *pseudo*-OOT: Home Credit has
no application dates, so it is a random stratified holdout, not a temporal one.

The split runs on SK_ID_CURR before any step that learns from data. Input order does not
matter: IDs are sorted first, so the same population and seed always give the same split.
"""

from __future__ import annotations

import hashlib

import pandas as pd
from sklearn.model_selection import train_test_split

from acr.core.schema import ID, TARGET

PARTITIONS = ("train", "test", "oot")


def stratified_sample(df: pd.DataFrame, frac: float, seed: int) -> pd.DataFrame:
    if frac >= 1.0:
        return df
    keep, _ = train_test_split(df, train_size=frac, stratify=df[TARGET], random_state=seed)
    return keep


def split_population(
    ids: pd.DataFrame,
    oot_frac: float,
    test_frac_of_dev: float,
    seed: int,
    sample_frac: float | None = None,
) -> pd.DataFrame:
    """Return ``[SK_ID_CURR, TARGET, partition]``, sorted by ID."""
    df = ids[[ID, TARGET]].sort_values(ID).reset_index(drop=True)
    if df[ID].duplicated().any():
        raise ValueError(f"duplicate {ID} in split input")
    if sample_frac is not None:
        df = stratified_sample(df, sample_frac, seed).sort_values(ID).reset_index(drop=True)
    dev, oot = train_test_split(df, test_size=oot_frac, stratify=df[TARGET], random_state=seed)
    train, test = train_test_split(
        dev, test_size=test_frac_of_dev, stratify=dev[TARGET], random_state=seed
    )
    out = pd.concat([train.assign(partition="train"), test.assign(partition="test"),
                     oot.assign(partition="oot")])  # fmt: skip
    out["partition"] = pd.Categorical(out["partition"], categories=PARTITIONS)
    return out.sort_values(ID).reset_index(drop=True)


def summary(assign: pd.DataFrame) -> dict:
    n = len(assign)
    g = assign.groupby("partition", observed=False)[TARGET]
    counts = g.size().to_dict()
    return {
        "population": n,
        "counts": {k: int(v) for k, v in counts.items()},
        "proportions": {k: round(v / n, 6) for k, v in counts.items()},
        "target_rate": {k: round(float(v), 6) for k, v in g.mean().to_dict().items()},
        "overall_target_rate": round(float(assign[TARGET].mean()), 6),
        "assignment_hash": hashlib.sha256(
            pd.util.hash_pandas_object(assign[[ID, "partition"]], index=False).values.tobytes()
        ).hexdigest()[:16],
    }

In [19]:
from acr.nodes.split import split

step(split, state)
s = state["split"]
display(pd.DataFrame({"count": s["counts"], "share": s["proportions"], "default rate": s["target_rate"]}))
print("assignment hash (same seed + same population = same hash):", s["assignment_hash"])

split: wrote ['split']


,count,share,default rate
train,17297,0.562486,0.080708
test,5766,0.187506,0.080645
oot,7688,0.250008,0.080775


assignment hash (same seed + same population = same hash): 57522e9fae1374f6


## 7. The four leg agents (nodes `leg_application`, `leg_bureau`, `leg_behavioural`, `leg_previous`)

One agent per data leg, mirroring a multi-leg PD design. In the graph they run **in
parallel** after `split`. Each aggregates its tables to one row per customer (`SK_ID_CURR`),
runs leg-level quality checks, and writes a feature table plus a **feature dictionary**
(name, definition, source table, leg, role).

**Leakage.** Leg features are per-customer aggregations of that customer's *own* records.
Leg builders are never given `TARGET`, so they can run on all partitions. Tests confirm that
features are unchanged when targets are shuffled.

**Protected characteristics.** `CODE_GENDER` and `NAME_FAMILY_STATUS` have role
`segment_only`: they're used for fairness analysis and never offered to the model.

In [20]:
import acr.core.legs.common
show(acr.core.legs.common)

"""Shared leg utilities: feature dictionary, safe ratios and leg-level quality checks.

Leg transforms are per-customer aggregations of that customer's own records. They learn
nothing from other customers or from TARGET (leg builders are never given TARGET), so they
can run on all partitions without leakage. Anything that learns from data (binning, WoE,
selection, calibration) happens later, on train only.
"""

from __future__ import annotations

from dataclasses import asdict, dataclass

import numpy as np
import pandas as pd

from acr.core.schema import DAYS_SENTINEL, ID

DAYS_PER_MONTH = 30.4375
DAYS_PER_YEAR = 365.25


@dataclass(frozen=True)
class FeatureDef:
    name: str
    definition: str
    source_table: str
    leg: str
    kind: str = "numeric"  # numeric | categorical
    role: str = "candidate"  # candidate | segment_only


class FeatureDict:
    def __init__(self, leg: str) -> None:
        self.leg = leg
        self._defs: dict[str, FeatureDef] = {}

    def add(self, name: str, definition: str, source: str, kind: str = "numeric",
            role: str = "candidate") -> str:  # fmt: skip
        if name in self._defs:
            raise ValueError(f"duplicate feature {name}")
        self._defs[name] = FeatureDef(name, definition, source, self.leg, kind, role)
        return name

    def frame(self) -> pd.DataFrame:
        return pd.DataFrame([asdict(d) for d in self._defs.values()])

    def names(self) -> list[str]:
        return list(self._defs)


@dataclass
class LegResult:
    features: pd.DataFrame  # indexed by SK_ID_CURR, one row per requested ID
    dictionary: pd.DataFrame
    has_record: pd.Series  # bool per ID: customer has any source record


def ratio(num: pd.Series, den: pd.Series) -> pd.Series:
    """num / den with zero or missing denominators giving NaN, never inf."""
    num = num.astype("float64")
    den = den.astype("float64")
    return (num / den.where(den != 0)).replace([np.inf, -np.inf], np.nan)


def clean_days(s: pd.Series) -> pd.Series:
    """Replace the 365243 sentinel with NaN."""
    s = s.astype("float64")
    return s.where(s != DAYS_SENTINEL)


def finalise(features: pd.DataFrame, fd: FeatureDict, ids: np.ndarray,
             count_cols: list[str] = ()) -> pd.DataFrame:  # fmt: skip
    """Reindex to all requested IDs, zero-fill count features, check against the dictionary."""
    out = features.reindex(pd.Index(ids, name=ID))
    for c in count_cols:
        out[c] = out[c].fillna(0)
    expected, got = fd.names(), list(out.columns)
    if sorted(expected) != sorted(got):
        raise ValueError(f"{fd.leg}: dictionary/feature mismatch "
                         f"{sorted(set(expected) ^ set(got))}")  # fmt: skip
    num = out.select_dtypes("number").columns
    out[num] = out[num].astype("float32")
    return out[expected]


def leg_quality(res: LegResult) -> dict:
    f = res.features
    num = f.select_dtypes("number")
    all_null = [c for c in f.columns if f[c].isna().all()]
    constant = [c for c in f.columns if f[c].nunique(dropna=True) <= 1 and c not in all_null]
    issues = []
    if not f.index.is_unique:
        issues.append({"severity": "critical", "check": "id_unique", "detail": "duplicate IDs"})
    if np.isinf(num.to_numpy(dtype="float64", na_value=np.nan)).any():
        issues.append({"severity": "high", "check": "no_inf", "detail": "infinite values"})
    if all_null:
        issues.append({"severity": "medium", "check": "all_null", "detail": all_null})
    if constant:
        issues.append({"severity": "low", "check": "constant", "detail": constant})
    return {
        "rows": len(f),
        "n_features": f.shape[1],
        "coverage": round(float(res.has_record.mean()), 6),
        "missing_share_mean": round(float(f.isna().mean().mean()), 6),
        "issues": issues,
        "status": "fail" if any(i["severity"] == "critical" for i in issues) else "pass",
    }

In [21]:
import acr.core.legs.application
show(acr.core.legs.application)

"""Application leg: applicant and loan attributes at the point of application."""

from __future__ import annotations

import numpy as np
import pandas as pd

from acr.core.io import read
from acr.core.legs.common import (
    DAYS_PER_YEAR,
    FeatureDict,
    LegResult,
    clean_days,
    finalise,
    ratio,
)  # fmt: skip
from acr.core.schema import APP_CATEGORICAL, APP_NUMERIC, APP_PROTECTED, DAYS_SENTINEL, ID

SRC = "application_train"

RAW_NUMERIC_DEFS = {
    "EXT_SOURCE_1": "External normalised score 1 (provider undisclosed by Home Credit)",
    "EXT_SOURCE_2": "External normalised score 2 (provider undisclosed by Home Credit)",
    "EXT_SOURCE_3": "External normalised score 3 (provider undisclosed by Home Credit)",
    "AMT_INCOME_TOTAL": "Declared annual income",
    "AMT_CREDIT": "Credit amount of the loan",
    "AMT_ANNUITY": "Loan annuity (periodic instalment)",
    "AMT_GOODS_PRICE": "Price of goods the loan finances",
    "CNT_CHILDREN": "Number of children",
    "CNT_FAM_MEMBERS": "Number of family members",
    "REGION_POPULATION_RELATIVE": "Normalised population of the client's region",
    "REGION_RATING_CLIENT_W_CITY": "Home Credit's rating of the client's region and city (1-3)",
    "OWN_CAR_AGE": "Age of the client's car in years",
    "AMT_REQ_CREDIT_BUREAU_MON": "Bureau enquiries in the month before application",
    "AMT_REQ_CREDIT_BUREAU_QRT": "Bureau enquiries in the 3 months before application",
    "AMT_REQ_CREDIT_BUREAU_YEAR": "Bureau enquiries in the year before application",
    "DEF_30_CNT_SOCIAL_CIRCLE": "Social-circle contacts 30+ DPD",
    "DEF_60_CNT_SOCIAL_CIRCLE": "Social-circle contacts 60+ DPD",
    "OBS_30_CNT_SOCIAL_CIRCLE": "Social-circle contacts with observable 30 DPD history",
    "FLAG_DOCUMENT_3": "Provided document 3",
    "REG_CITY_NOT_WORK_CITY": "Registered city differs from work city",
    "REG_CITY_NOT_LIVE_CITY": "Registered city differs from living city",
    "FLAG_WORK_PHONE": "Provided a work phone",
    "FLAG_PHONE": "Provided a home phone",
    "TOTALAREA_MODE": "Normalised total area of the client's building (mode)",
}
CATEGORICAL_DEFS = {
    "NAME_CONTRACT_TYPE": "Contract type (cash or revolving)",
    "FLAG_OWN_CAR": "Owns a car (Y/N)",
    "FLAG_OWN_REALTY": "Owns real estate (Y/N)",
    "NAME_INCOME_TYPE": "Income type (working, pensioner, ...)",
    "NAME_EDUCATION_TYPE": "Highest education level",
    "NAME_HOUSING_TYPE": "Housing situation",
    "OCCUPATION_TYPE": "Occupation",
    "ORGANIZATION_TYPE": "Employer organisation type",
}
PROTECTED_DEFS = {
    "CODE_GENDER": "Gender. Protected characteristic: segment and fairness analysis only",
    "NAME_FAMILY_STATUS": "Marital status. Protected characteristic: segment and fairness analysis only",
}


def build(paths: dict[str, str], ids: np.ndarray) -> LegResult:
    cols = [ID, *APP_NUMERIC, *APP_CATEGORICAL, *APP_PROTECTED]
    # Labelled applicants (application_train) plus, when present, the unlabelled Kaggle
    # scoring set (application_test). Neither read includes the default flag.
    frames = [read(paths["application"], cols, ids)]
    if paths.get("application_test"):
        frames.append(read(paths["application_test"], cols, ids))
    app = pd.concat([f for f in frames if len(f)], ignore_index=True).set_index(ID)
    fd = FeatureDict("application")
    f = pd.DataFrame(index=app.index)

    skip = {"DAYS_BIRTH", "DAYS_EMPLOYED", "DAYS_REGISTRATION", "DAYS_ID_PUBLISH",
            "DAYS_LAST_PHONE_CHANGE"}  # fmt: skip
    for col in APP_NUMERIC:
        if col not in skip:
            f[fd.add(f"APP_{col}", RAW_NUMERIC_DEFS[col], SRC)] = app[col].astype("float64")

    emp_raw = app["DAYS_EMPLOYED"].astype("float64")
    emp = clean_days(emp_raw)
    age = -app["DAYS_BIRTH"].astype("float64") / DAYS_PER_YEAR
    f[fd.add("APP_AGE_YEARS", "Age in years at application (-DAYS_BIRTH / 365.25)", SRC)] = age
    f[fd.add("APP_EMPLOYED_YEARS", "Years in current employment; 365243 sentinel set to miss

In [22]:
import acr.core.legs.bureau
show(acr.core.legs.bureau)

"""Bureau leg: credit with other lenders (bureau) and its monthly status history (bureau_balance)."""

from __future__ import annotations

import numpy as np
import pandas as pd

from acr.core.io import read
from acr.core.legs.common import DAYS_PER_MONTH, FeatureDict, LegResult, finalise, ratio
from acr.core.schema import ID

B, BB = "bureau", "bureau_balance"
# bureau_balance STATUS: 0 = no DPD, 1 = 1-30, 2 = 31-60, 3 = 61-90, 4 = 91-120,
# 5 = 120+ or written off; C = closed, X = unknown (both treated as missing bucket).
STATUS_TO_BUCKET = {"0": 0, "1": 1, "2": 2, "3": 3, "4": 4, "5": 5}


def dpd_buckets(status: pd.Series) -> pd.Series:
    return status.astype("string").map(STATUS_TO_BUCKET).astype("float32")


def build(paths: dict[str, str], ids: np.ndarray) -> LegResult:
    cols = ["SK_ID_BUREAU", ID, "CREDIT_ACTIVE", "DAYS_CREDIT", "CREDIT_DAY_OVERDUE",
            "DAYS_CREDIT_ENDDATE", "AMT_CREDIT_MAX_OVERDUE", "CNT_CREDIT_PROLONG",
            "AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT", "AMT_CREDIT_SUM_OVERDUE", "CREDIT_TYPE"]  # fmt: skip
    b = read(paths["bureau"], cols, ids)
    fd = FeatureDict("bureau")
    active = b["CREDIT_ACTIVE"].astype("string") == "Active"
    ctype = b["CREDIT_TYPE"].astype("string")
    g = b.groupby(ID)
    ga = b[active].groupby(ID)
    f = pd.DataFrame(index=pd.Index(g.size().index, name=ID))
    counts = []

    def cnt(name, definition, s):
        counts.append(fd.add(name, definition, B))
        f[name] = s

    cnt("BUR_COUNT", "Number of bureau credits", g.size())
    cnt("BUR_ACTIVE_COUNT", "Number of active bureau credits", active.groupby(b[ID]).sum())
    cnt(
        "BUR_OPENED_12M_COUNT",
        "Bureau credits opened in the 12 months before application",
        (b["DAYS_CREDIT"] >= -365).groupby(b[ID]).sum(),
    )
    cnt(
        "BUR_MICROLOAN_COUNT",
        "Number of bureau microloans",
        (ctype == "Microloan").groupby(b[ID]).sum(),
    )
    cnt(
        "BUR_CREDIT_CARD_COUNT",
        "Number of bureau credit cards",
        (ctype == "Credit card").groupby(b[ID]).sum(),
    )
    cnt(
        "BUR_PROLONG_SUM",
        "Total number of bureau credit prolongations",
        g["CNT_CREDIT_PROLONG"].sum(),
    )
    f[fd.add("BUR_ACTIVE_SHARE", "Active / all bureau credits", B)] = ratio(
        f["BUR_ACTIVE_COUNT"], f["BUR_COUNT"]
    )
    f[fd.add("BUR_MONTHS_SINCE_FIRST", "Months since the oldest bureau credit was opened", B)] = (
        -g["DAYS_CREDIT"].min() / DAYS_PER_MONTH
    )
    f[fd.add("BUR_MONTHS_SINCE_LAST", "Months since the newest bureau credit was opened", B)] = (
        -g["DAYS_CREDIT"].max() / DAYS_PER_MONTH
    )
    debt = ga["AMT_CREDIT_SUM_DEBT"].sum()
    limit = ga["AMT_CREDIT_SUM"].sum()
    f[fd.add("BUR_ACTIVE_DEBT_SUM", "Outstanding debt on active bureau credits", B)] = debt
    f[fd.add("BUR_ACTIVE_DEBT_TO_CREDIT", "Active outstanding debt / active credit amount", B)] = (
        ratio(debt.reindex(f.index), limit.reindex(f.index))
    )
    f[fd.add("BUR_OVERDUE_SUM", "Current overdue amount across bureau credits", B)] = g[
        "AMT_CREDIT_SUM_OVERDUE"
    ].sum()
    f[fd.add("BUR_DAYS_OVERDUE_MAX", "Current maximum days overdue on any bureau credit", B)] = g[
        "CREDIT_DAY_OVERDUE"
    ].max()
    f[fd.add("BUR_AMT_MAX_OVERDUE_MAX", "Largest overdue amount ever on any bureau credit", B)] = g[
        "AMT_CREDIT_MAX_OVERDUE"
    ].max()
    f[
        fd.add(
            "BUR_ACTIVE_REMAINING_MONTHS_MEAN", "Mean remaining months on active bureau credits", B
        )
    ] = ga["DAYS_CREDIT_ENDDATE"].mean() / DAYS_PER_MONTH

    # bureau_balance: monthly DPD buckets per bureau credit, rolled up to the customer.
    bb = read(
        paths["bureau_balance"],
        ["SK_ID_BUREAU", "MONTHS_BALANCE", "STATUS"],
        ids=b["SK_ID_BUREAU"].unique(),
        id_col="SK_ID_BUREAU",
    )
    bb["bucket"] = dpd_buckets(bb.pop("STATUS"))
    bb = bb.merge(b[["SK_ID_BUREAU", ID]], on="SK_ID_BUREAU", how="inn

In [23]:
import acr.core.legs.behavioural
show(acr.core.legs.behavioural)

"""Behavioural leg: repayment behaviour on the customer's earlier Home Credit loans.

Sources: installments_payments, POS_CASH_balance, credit_card_balance.
"""

from __future__ import annotations

import numpy as np
import pandas as pd

from acr.core.io import read
from acr.core.legs.common import FeatureDict, LegResult, finalise, ratio
from acr.core.schema import ID

INS, POS, CC = "installments_payments", "POS_CASH_balance", "credit_card_balance"
INST_KEY = ["SK_ID_PREV", "NUM_INSTALMENT_VERSION", "NUM_INSTALMENT_NUMBER"]


def instalment_level(ins: pd.DataFrame) -> pd.DataFrame:
    """Collapse payment rows to one row per scheduled instalment.

    An instalment can be paid in several part-payments. Days past due is measured from the
    due date to the *last* payment, so a partly paid instalment counts as late until it is
    fully settled. Instalments with no payment record have unknown DPD (NaN).
    """
    agg = ins.groupby(INST_KEY, observed=True, sort=False).agg(
        cust=(ID, "first"), due=("DAYS_INSTALMENT", "min"), amt=("AMT_INSTALMENT", "max"),
        paid=("AMT_PAYMENT", "sum"), last_pay=("DAYS_ENTRY_PAYMENT", "max"),
    )  # fmt: skip
    agg["dpd"] = (agg["last_pay"] - agg["due"]).clip(lower=0)
    agg["early"] = (agg["due"] - agg["last_pay"]).clip(lower=0)
    agg["paid_ratio"] = ratio(agg["paid"], agg["amt"])
    agg["shortfall"] = (agg["amt"] - agg["paid"]).clip(lower=0)
    return agg.reset_index(drop=True).rename(columns={"cust": ID})


def build(paths: dict[str, str], ids: np.ndarray) -> LegResult:
    fd = FeatureDict("behavioural")
    idx = pd.Index(ids, name=ID)
    f = pd.DataFrame(index=idx)
    counts: list[str] = []

    # --- installments ---
    ins = read(paths["installments"], [*INST_KEY, ID, "DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT",
                                       "AMT_INSTALMENT", "AMT_PAYMENT"], ids)  # fmt: skip
    il = instalment_level(ins)
    del ins
    g = il.groupby(ID)
    recent = il[il["due"] >= -365]
    gr = recent.groupby(ID)
    counts.append(fd.add("BEH_INS_COUNT", "Scheduled instalments on prior Home Credit loans", INS))
    f["BEH_INS_COUNT"] = g.size()
    f[fd.add("BEH_INS_LATE_SHARE", "Share of instalments paid after the due date", INS)] = (
        (il["dpd"] > 0).groupby(il[ID]).mean()
    )
    f[fd.add("BEH_INS_DPD_MAX", "Maximum days past due on any instalment", INS)] = g["dpd"].max()
    f[fd.add("BEH_INS_DPD_MEAN", "Mean days past due per instalment", INS)] = g["dpd"].mean()
    f[
        fd.add("BEH_INS_DPD_MAX_12M", "Maximum DPD on instalments due in the last 12 months", INS)
    ] = gr["dpd"].max()
    counts.append(
        fd.add(
            "BEH_INS_30DPD_COUNT_12M",
            "Instalments more than 30 DPD, due in the last 12 months",
            INS,
        )
    )
    f["BEH_INS_30DPD_COUNT_12M"] = (recent["dpd"] > 30).groupby(recent[ID]).sum()
    f[
        fd.add(
            "BEH_INS_90DPD_12M_FLAG",
            "1 if any instalment due in the last 12 months was more than 90 DPD",
            INS,
        )
    ] = (gr["dpd"].max() > 90).astype(float)
    f[fd.add("BEH_INS_PAID_RATIO_MEAN", "Mean of paid / scheduled amount per instalment", INS)] = g[
        "paid_ratio"
    ].mean()
    f[
        fd.add(
            "BEH_INS_UNDERPAID_SHARE",
            "Share of instalments paid below 99% of the scheduled amount",
            INS,
        )
    ] = (il["paid_ratio"] < 0.99).groupby(il[ID]).mean()
    f[fd.add("BEH_INS_SHORTFALL_SUM", "Total unpaid amount across instalments", INS)] = g[
        "shortfall"
    ].sum()
    f[fd.add("BEH_INS_EARLY_DAYS_MEAN", "Mean days paid before the due date", INS)] = g[
        "early"
    ].mean()
    has_ins = idx.isin(g.size().index)
    del il, recent

    # --- POS / cash loans, monthly ---
    pos = read(paths["pos_cash"], ["SK_ID_PREV", ID, "MONTHS_BALANCE", "CNT_INSTALMENT_FUTURE",
                                   "NAME_CONTRACT_STATUS", "SK_DPD_DEF"], ids)  # fmt: skip
    gp = pos.g

In [24]:
import acr.core.legs.previous
show(acr.core.legs.previous)

"""Previous applications leg: the customer's earlier applications to Home Credit."""

from __future__ import annotations

import numpy as np
import pandas as pd

from acr.core.io import read
from acr.core.legs.common import DAYS_PER_MONTH, FeatureDict, LegResult, clean_days, finalise, ratio
from acr.core.schema import ID

SRC = "previous_application"


def build(paths: dict[str, str], ids: np.ndarray) -> LegResult:
    cols = ["SK_ID_PREV", ID, "NAME_CONTRACT_TYPE", "AMT_ANNUITY", "AMT_APPLICATION",
            "AMT_CREDIT", "RATE_DOWN_PAYMENT", "NAME_CONTRACT_STATUS", "DAYS_DECISION",
            "CNT_PAYMENT", "NAME_YIELD_GROUP", "DAYS_TERMINATION"]  # fmt: skip
    p = read(paths["previous"], cols, ids)
    fd = FeatureDict("previous")
    status = p["NAME_CONTRACT_STATUS"].astype("string")
    ctype = p["NAME_CONTRACT_TYPE"].astype("string")
    approved, refused = status == "Approved", status == "Refused"
    recent = p["DAYS_DECISION"] >= -365
    term = p["DAYS_TERMINATION"].astype("float64")
    # 365243 in DAYS_TERMINATION means "no termination date yet"; positive means in the future.
    still_running = approved & (term.isna() | (term == 365243) | (term > 0))
    g = p.groupby(ID)
    by = p[ID]
    f = pd.DataFrame(index=pd.Index(g.size().index, name=ID))
    counts: list[str] = []

    def cnt(name, definition, s):
        counts.append(fd.add(name, definition, SRC))
        f[name] = s

    cnt("PRV_COUNT", "Number of previous applications", g.size())
    cnt("PRV_APPROVED_COUNT", "Approved previous applications", approved.groupby(by).sum())
    cnt("PRV_REFUSED_COUNT", "Refused previous applications", refused.groupby(by).sum())
    cnt(
        "PRV_COUNT_12M",
        "Previous applications decided in the last 12 months",
        recent.groupby(by).sum(),
    )
    cnt("PRV_REFUSED_12M", "Refusals in the last 12 months", (refused & recent).groupby(by).sum())
    cnt(
        "PRV_RUNNING_COUNT",
        "Approved previous loans with no termination yet",
        still_running.groupby(by).sum(),
    )
    cnt(
        "PRV_REVOLVING_COUNT",
        "Previous revolving-loan applications",
        (ctype == "Revolving loans").groupby(by).sum(),
    )
    f[fd.add("PRV_REFUSED_SHARE", "Refused / all previous applications", SRC)] = ratio(
        f["PRV_REFUSED_COUNT"], f["PRV_COUNT"]
    )
    f[fd.add("PRV_CANCELED_SHARE", "Cancelled / all previous applications", SRC)] = ratio(
        (status == "Canceled").groupby(by).sum(), f["PRV_COUNT"]
    )
    dec = clean_days(p["DAYS_DECISION"])
    f[fd.add("PRV_MONTHS_SINCE_LAST", "Months since the latest previous decision", SRC)] = (
        -dec.groupby(by).max() / DAYS_PER_MONTH
    )
    f[fd.add("PRV_MONTHS_SINCE_REFUSAL", "Months since the latest refusal", SRC)] = (
        -dec[refused].groupby(by[refused]).max() / DAYS_PER_MONTH
    )
    a = p[approved]
    ga = a.groupby(ID)
    f[
        fd.add(
            "PRV_CREDIT_TO_APPLICATION_MEAN",
            "Mean granted / requested amount, approved applications",
            SRC,
        )
    ] = ratio(a["AMT_CREDIT"], a["AMT_APPLICATION"]).groupby(a[ID]).mean()
    f[fd.add("PRV_AMT_CREDIT_MEAN", "Mean credit amount, approved applications", SRC)] = ga[
        "AMT_CREDIT"
    ].mean()
    f[fd.add("PRV_ANNUITY_MEAN", "Mean annuity, approved applications", SRC)] = ga[
        "AMT_ANNUITY"
    ].mean()
    f[fd.add("PRV_CNT_PAYMENT_MEAN", "Mean term in instalments, approved applications", SRC)] = ga[
        "CNT_PAYMENT"
    ].mean()
    f[fd.add("PRV_DOWN_PAYMENT_RATE_MEAN", "Mean down-payment rate", SRC)] = g[
        "RATE_DOWN_PAYMENT"
    ].mean()
    f[
        fd.add("PRV_HIGH_YIELD_SHARE", "Share of applications in the high interest-rate group", SRC)
    ] = (p["NAME_YIELD_GROUP"].astype("string") == "high").groupby(by).mean()
    f[fd.add("PRV_CASH_LOAN_SHARE", "Share of applications that were cash loans", SRC)] = (
        (ctype == "Cash loans").groupby(by).mean()
    )

    has = pd.Series(pd.Index(

In [25]:
# The thin node wrapper shared by all four legs
import acr.nodes.legs._base
show(acr.nodes.legs._base)

"""Shared implementation for the four leg nodes."""

from __future__ import annotations

import time
from collections.abc import Callable, Mapping

import numpy as np
import pandas as pd

from acr.core.io import read
from acr.core.legs.common import LegResult, leg_quality
from acr.core.schema import ID
from acr.nodes._util import artefact_dir


def run_leg(view: Mapping, leg: str, build: Callable) -> dict:
    t0 = time.perf_counter()
    paths = {k: v["path"] for k, v in view["dq"]["tables"].items()}
    ids = read(view["split"]["assignment_path"], [ID])[ID].to_numpy()
    ext = view["split"].get("external")
    if ext:  # unlabelled applicants get features too, so the model can score them
        ids = np.concatenate([ids, read(ext["path"], [ID])[ID].to_numpy()])
    res: LegResult = build(paths, ids)
    q = leg_quality(res)
    if q["status"] == "fail":
        raise ValueError(f"leg {leg} failed quality checks: {q['issues']}")
    out = artefact_dir(view, "legs", leg)
    res.features.to_parquet(out / "features.parquet")
    res.dictionary.to_csv(out / "dictionary.csv", index=False)
    roles = res.dictionary["role"].value_counts().to_dict()
    return {
        leg: {
            "features_path": str(out / "features.parquet"),
            "dictionary_path": str(out / "dictionary.csv"),
            "n_features": int(res.features.shape[1]),
            "roles": {k: int(v) for k, v in roles.items()},
            "dq": q,
            "seconds": round(time.perf_counter() - t0, 2),
        }
    }


def load_dictionary(path: str) -> pd.DataFrame:
    return pd.read_csv(path)

In [26]:
from acr.nodes.legs import LEG_NODES

# In the graph these four run in parallel; here we run them one after another.
for fn in LEG_NODES.values():
    step(fn, state)

summary = pd.DataFrame({
    leg: {"features": v["n_features"], "coverage": v["dq"]["coverage"],
          "mean missing share": v["dq"]["missing_share_mean"], "issues": len(v["dq"]["issues"]),
          "seconds": v["seconds"]}
    for leg, v in state["legs"].items()
}).T
display(summary)

leg_application: wrote ['legs']


leg_bureau: wrote ['legs']


leg_behavioural: wrote ['legs']


leg_previous: wrote ['legs']


,features,coverage,mean missing share,issues,seconds
application,49.0,1.000000,0.054334,0.0,0.17
bureau,22.0,0.856337,0.309438,0.0,0.59
behavioural,25.0,0.956042,0.271356,0.0,0.90
previous,18.0,0.952000,0.067719,0.0,0.23


In [27]:
# A slice of the feature dictionary (definitions only, no data)
dictionary = pd.concat(pd.read_csv(v["dictionary_path"]) for v in state["legs"].values())
display(dictionary.groupby("leg").head(4)[["leg", "name", "definition", "source_table", "role"]])

,leg,name,definition,source_table,role
0,application,APP_EXT_SOURCE_1,External normalised score 1 (provider undisclo...,application_train,candidate
1,application,APP_EXT_SOURCE_2,External normalised score 2 (provider undisclo...,application_train,candidate
2,application,APP_EXT_SOURCE_3,External normalised score 3 (provider undisclo...,application_train,candidate
3,application,APP_AMT_INCOME_TOTAL,Declared annual income,application_train,candidate
0,bureau,BUR_COUNT,Number of bureau credits,bureau,candidate
1,bureau,BUR_ACTIVE_COUNT,Number of active bureau credits,bureau,candidate
2,bureau,BUR_OPENED_12M_COUNT,Bureau credits opened in the 12 months before ...,bureau,candidate
3,bureau,BUR_MICROLOAN_COUNT,Number of bureau microloans,bureau,candidate
0,behavioural,BEH_INS_COUNT,Scheduled instalments on prior Home Credit loans,installments_payments,candidate
1,behavioural,BEH_INS_LATE_SHARE,Share of instalments paid after the due date,installments_payments,candidate


## 8. Feature selection (node `select`): train only

The four legs are joined, and scorecard features are selected **using train only**. Test is
used for one thing: the train-vs-test stability (CSI) screen, which uses test *features*,
never test targets.

Binning uses **optbinning** (monotonic WoE bins). WoE convention: `ln(%good / %bad)`, so a
higher WoE means lower risk, and every scorecard coefficient should be negative.

The stages, each logged with a reason:

1. role (protected characteristics)
2. missingness above 95%
3. single bin
4. IV below 0.02, or above 0.5 (possible leakage)
5. CSI above 0.10
6. |correlation| above 0.7
7. VIF above 5
8. cap at 15 features
9. backward elimination: wrong coefficient sign, then p-value above 0.05

In [28]:
import acr.core.guards
show(acr.core.guards)   # every fitter calls this: non-train rows raise LeakageError

"""Leakage guards. Fitting functions call these so non-train rows can never reach a fitter."""

from __future__ import annotations

import pandas as pd


class LeakageError(RuntimeError):
    """Rows from a partition other than the one allowed were passed to a fitting step."""


def require_partition(frame: pd.DataFrame, assignment: pd.DataFrame, partition: str = "train"):
    """Raise unless every row of ``frame`` (indexed by SK_ID_CURR) belongs to ``partition``."""
    part = assignment.set_index("SK_ID_CURR")["partition"]
    found = part.reindex(frame.index)
    if found.isna().any():
        raise LeakageError(f"{int(found.isna().sum())} rows have no split assignment")
    bad = found[found != partition]
    if len(bad):
        raise LeakageError(
            f"{len(bad)} rows from {sorted(bad.astype(str).unique())} passed to a "
            f"{partition}-only fitting step"
        )
    return frame

In [29]:
import acr.core.binning
show(acr.core.binning)

"""Optimal WoE binning (optbinning), fitted on train only.

WoE convention (optbinning): WoE = ln(%non-events / %events), so a higher WoE means lower
risk. In a logistic regression of default on WoE, every coefficient should be negative.
Missing values get their own bin with its empirical WoE (optbinning's default of 0 is
overridden).
"""

from __future__ import annotations

import numpy as np
import pandas as pd
from optbinning import BinningProcess

from acr.core.metrics import psi

TRANSFORM_KW = {"metric_missing": "empirical", "metric_special": "empirical"}


def prepare(X: pd.DataFrame, categorical: list[str]) -> pd.DataFrame:
    """optbinning needs plain object strings for categoricals and float numerics."""
    X = X.copy()
    for c in X.columns:
        if c in categorical:
            X[c] = X[c].astype(object).where(X[c].notna(), np.nan)
        else:
            X[c] = X[c].astype("float64")
    return X


def fit_binning(X: pd.DataFrame, y: pd.Series, categorical: list[str], min_bin_size: float,
                max_n_bins: int) -> BinningProcess:  # fmt: skip
    numeric = [c for c in X.columns if c not in categorical]
    bp = BinningProcess(
        list(X.columns),
        categorical_variables=categorical,
        min_bin_size=min_bin_size,
        max_n_bins=max_n_bins,
        binning_fit_params={c: {"monotonic_trend": "auto_asc_desc"} for c in numeric},
    )
    bp.fit(prepare(X, categorical), y.to_numpy())
    return bp


def woe(bp: BinningProcess, X: pd.DataFrame, categorical: list[str]) -> pd.DataFrame:
    out = bp.transform(prepare(X, categorical), metric="woe", **TRANSFORM_KW)
    out.index = X.index
    return out


def bins(bp: BinningProcess, X: pd.DataFrame, categorical: list[str]) -> pd.DataFrame:
    out = bp.transform(prepare(X, categorical), metric="indices", **TRANSFORM_KW)
    out.index = X.index
    return out


def iv_table(bp: BinningProcess, features: list[str]) -> pd.DataFrame:
    rows = []
    for f in features:
        ob = bp.get_binned_variable(f)
        bt = ob.binning_table.build()
        rows.append({"feature": f, "iv": float(bt.loc["Totals", "IV"]),
                     "n_bins": int(len(bt) - 3), "status": ob.status})  # fmt: skip
    return pd.DataFrame(rows)


def binning_tables(bp: BinningProcess, features: list[str]) -> pd.DataFrame:
    """Long table of every bin for every feature: for documentation and the points table."""
    frames = []
    for f in features:
        bt = bp.get_binned_variable(f).binning_table.build().drop(index="Totals")
        bt = bt.reset_index(names="bin_index")
        bt["Bin"] = bt["Bin"].map(
            lambda b: ", ".join(map(str, b)) if isinstance(b, (list, np.ndarray)) else str(b)
        )
        frames.append(
            bt.assign(feature=f)[
                ["feature", "bin_index", "Bin", "Count", "Event rate", "WoE", "IV"]
            ]
        )
    return pd.concat(frames, ignore_index=True).rename(
        columns={
            "Bin": "bin",
            "Count": "count",
            "Event rate": "event_rate",
            "WoE": "woe",
            "IV": "iv",
        }
    )


def csi(bins_a: pd.DataFrame, bins_b: pd.DataFrame) -> dict[str, float]:
    """Characteristic stability index per feature over the fitted bins."""
    return {c: psi(bins_a[c], bins_b[c]) for c in bins_a.columns}

In [30]:
import acr.core.selection
show(acr.core.selection)

"""Feature selection for the champion scorecard. Every learned step uses train only.

Stages, in order. Each exclusion is logged with the stage, reason and metric value:

1. role       protected characteristics are segment-only (never candidates)
2. missing    missing share on train above ``missing_max``
3. binning    optimal binning yields a single bin (no discrimination)
4. iv         IV below ``iv_min``, or above ``iv_suspicious`` (possible leakage)
5. csi        train-vs-test characteristic stability above ``csi_max`` (uses test *features* only)
6. corr       |Pearson correlation of WoE| above ``corr_max`` with a higher-IV kept feature
7. vif        iterative removal of the highest VIF above ``vif_max``
8. cap        beyond the top ``max_features`` by IV
9. model      backward elimination: wrong coefficient sign, then p-value above ``p_value_max``
"""

from __future__ import annotations

from dataclasses import dataclass, field

import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

from acr.config import FeatureCfg
from acr.core import binning as B
from acr.core.guards import require_partition


@dataclass
class SelectionResult:
    candidates: list[str]
    categorical: list[str]
    selected: list[str]
    exclusions: pd.DataFrame
    iv: pd.DataFrame
    csi: dict[str, float]
    binning: object  # fitted optbinning BinningProcess (train only)
    screened: list[str] = field(default_factory=list)  # survivors of stages 1-5
    vif: dict[str, float] = field(default_factory=dict)


def _vif(W: pd.DataFrame) -> pd.Series:
    X = sm.add_constant(W.to_numpy(dtype="float64"), has_constant="add")
    return pd.Series(
        [variance_inflation_factor(X, i + 1) for i in range(W.shape[1])], index=W.columns
    )


def logit_fit(W: pd.DataFrame, y: pd.Series):
    return sm.Logit(y.to_numpy(dtype="float64"), sm.add_constant(W.to_numpy(dtype="float64"),
                    has_constant="add")).fit(disp=0, maxiter=200)  # fmt: skip


def backward_eliminate(W: pd.DataFrame, y: pd.Series, p_max: float,
                       log: list[dict], stage: str = "model") -> list[str]:  # fmt: skip
    """Drop wrong-sign coefficients first (WoE coefficients must be negative), then the least
    significant feature, one at a time, refitting after every drop."""
    feats = list(W.columns)
    while feats:
        res = logit_fit(W[feats], y)
        coefs = pd.Series(res.params[1:], index=feats)
        pvals = pd.Series(res.pvalues[1:], index=feats)
        wrong = coefs[coefs >= 0]
        if len(wrong):
            f = wrong.idxmax()
            log.append({"feature": f, "stage": stage, "reason": "wrong coefficient sign "
                        "(positive on WoE)", "value": round(float(coefs[f]), 6)})  # fmt: skip
        elif pvals.max() > p_max:
            f = pvals.idxmax()
            log.append({"feature": f, "stage": stage, "reason": f"p-value above {p_max}",
                        "value": round(float(pvals[f]), 6)})  # fmt: skip
        else:
            return feats
        feats.remove(f)
    return feats


def select_features(features: pd.DataFrame, dictionary: pd.DataFrame, assignment: pd.DataFrame,
                    cfg: FeatureCfg, binning=None) -> SelectionResult:  # fmt: skip
    """``features`` is indexed by SK_ID_CURR and covers all partitions; ``assignment`` holds
    SK_ID_CURR, TARGET and partition. Targets are read for train only.

    ``binning`` lets offline tuning reuse a BinningProcess already fitted on the same train
    rows with the same bin settings; pipeline runs always fit it here."""
    a = assignment.set_index("SK_ID_CURR")
    train_ids = a.index[a["partition"] == "train"]
    test_ids = a.index[a["partition"] == "test"]
    Xtr = require_partition(features.loc[train_ids], assignment, "train")
    ytr = a.loc[train_ids, "TARGET"]
    log: list[dict] = []

    d = dictionary.set_index("name")
    for f in d.index[d["role"] != "c

In [31]:
import acr.nodes.select
show(acr.nodes.select)

"""Node 4: feature assembly and selection (train only)."""

from __future__ import annotations

import json
import pickle

import pandas as pd

from acr.audit import node
from acr.core import binning as B
from acr.core.selection import select_features
from acr.nodes._util import artefact_dir, cfg


@node(
    "select",
    role="Join the four legs; select scorecard features on train only by missingness, IV, "
    "train-vs-test CSI, correlation, VIF, a size cap and coefficient sign/significance; log "
    "every exclusion with a reason.",
    reads=["config", "split", "legs"],
    writes=["selection"],
)
def select(view):
    c = cfg(view)
    legs = view["legs"]
    order = ["application", "bureau", "behavioural", "previous"]
    features = pd.concat([pd.read_parquet(legs[k]["features_path"]) for k in order], axis=1)
    dictionary = pd.concat([pd.read_csv(legs[k]["dictionary_path"]) for k in order],
                           ignore_index=True)  # fmt: skip
    assignment = pd.read_parquet(view["split"]["assignment_path"])
    sel = select_features(features, dictionary, assignment, c.base.features)

    out = artefact_dir(view, "selection")
    features.to_parquet(out / "features_all.parquet")
    dictionary.to_csv(out / "feature_dictionary.csv", index=False)
    with open(out / "binning_process.pkl", "wb") as f:
        pickle.dump(sel.binning, f)
    sel.iv.sort_values("iv", ascending=False).to_csv(out / "iv_table.csv", index=False)
    sel.exclusions.to_csv(out / "exclusions.csv", index=False)
    B.binning_tables(sel.binning, list(sel.binning.variable_names)).to_csv(
        out / "binning_tables.csv", index=False)  # fmt: skip
    iv = sel.iv.set_index("feature")["iv"]
    meta = {
        "candidates": sel.candidates, "categorical": sel.categorical,
        "screened": sel.screened, "selected": sel.selected,
        "csi": {k: round(v, 6) for k, v in sel.csi.items()},
    }  # fmt: skip
    (out / "selection.json").write_text(json.dumps(meta, indent=2))
    stage_counts = sel.exclusions["stage"].value_counts().to_dict()
    leg_of = dictionary.set_index("name")["leg"]
    return {
        "selection": {
            "n_features_total": int(features.shape[1]),
            "n_candidates": len(sel.candidates),
            "n_screened": len(sel.screened),
            "n_selected": len(sel.selected),
            "selected": [
                {
                    "feature": f,
                    "leg": leg_of[f],
                    "iv": round(float(iv[f]), 4),
                    "csi": round(sel.csi[f], 4),
                    "vif": sel.vif.get(f),
                }
                for f in sel.selected
            ],  # fmt: skip
            "selected_by_leg": pd.Series([leg_of[f] for f in sel.selected])
            .value_counts()
            .to_dict(),
            "exclusions_by_stage": {k: int(v) for k, v in stage_counts.items()},
            "paths": {
                "features_all": str(out / "features_all.parquet"),
                "dictionary": str(out / "feature_dictionary.csv"),
                "binning": str(out / "binning_process.pkl"),
                "iv_table": str(out / "iv_table.csv"),
                "exclusions": str(out / "exclusions.csv"),
                "binning_tables": str(out / "binning_tables.csv"),
                "meta": str(out / "selection.json"),
            },
        }
    }

In [32]:
from acr.nodes.select import select

step(select, state)
sel = state["selection"]
print(f"{sel['n_features_total']} features -> {sel['n_candidates']} candidates -> "
      f"{sel['n_screened']} after screens -> {sel['n_selected']} selected")
print("exclusions by stage:", sel["exclusions_by_stage"])
display(pd.DataFrame(sel["selected"]))

select: wrote ['selection']
114 features -> 112 candidates -> 57 after screens -> 10 selected
exclusions by stage: {'iv': 43, 'cap': 31, 'binning': 12, 'corr': 11, 'model': 5, 'role': 2}


,feature,leg,iv,csi,vif
0,APP_EXT_SOURCE_3,application,0.3802,0.0009,1.1179
1,APP_EXT_SOURCE_2,application,0.3555,0.0007,1.0404
2,APP_EXT_SOURCE_1,application,0.1731,0.0005,1.0586
3,APP_OCCUPATION_TYPE,application,0.1204,0.0010,1.1119
4,BUR_MONTHS_SINCE_FIRST,bureau,0.1026,0.0009,1.1003
5,BEH_CC_UTIL_MEAN,behavioural,0.0998,0.0013,1.0405
6,BEH_INS_DPD_MAX_12M,behavioural,0.0833,0.0000,1.0365
7,APP_ORGANIZATION_TYPE,application,0.0807,0.0013,1.0817
8,BUR_ACTIVE_DEBT_TO_CREDIT,bureau,0.0759,0.0005,1.0372
9,APP_CREDIT_TO_GOODS,application,0.0720,0.0012,1.0149


In [33]:
# Every exclusion with its reason (first rows of each stage)
ex = pd.read_csv(sel["paths"]["exclusions"])
display(ex.groupby("stage").head(3)[["stage", "feature", "leg", "reason", "value"]])

,stage,feature,leg,reason,value
0,role,APP_CODE_GENDER,application,role is segment_only (protected characteristic),NaN
1,role,APP_NAME_FAMILY_STATUS,application,role is segment_only (protected characteristic),NaN
2,iv,APP_AMT_ANNUITY,application,IV below 0.02,0.012778
3,iv,APP_CNT_CHILDREN,application,IV below 0.02,0.001261
4,iv,APP_CNT_FAM_MEMBERS,application,IV below 0.02,0.000878
25,binning,BUR_MICROLOAN_COUNT,bureau,single bin after optimal binning,1.000000
27,binning,BUR_PROLONG_SUM,bureau,single bin after optimal binning,1.000000
29,binning,BUR_OVERDUE_SUM,bureau,single bin after optimal binning,1.000000
57,corr,BEH_CC_UTIL_MAX,behavioural,|corr| with BEH_CC_UTIL_MEAN above 0.7,0.759200
58,corr,APP_EMPLOYED_TO_AGE,application,|corr| with APP_EMPLOYED_YEARS above 0.7,0.910200


## 9. Modelling (node `model`): train only

* **Champion: monotone LightGBM** (`core/challenger.py`) on all candidate features, with
  **fixed** hyperparameters from config (tuned offline, §12) and **monotonic constraints** taken
  from each feature's train-only WoE trend. Early stopping uses a slice of train, never test or
  OOT. `model.champion` in config assigns the roles (decision D33).
* **Benchmark challenger and fallback: WoE logistic regression scorecard** (`core/scorecard.py`).
  It exports a coefficient table (with standard errors and p-values) and a **points table** using
  the PDO method (PDO 20, base score 600 at 50:1 odds). A higher score means lower risk.
* **Calibration** (`core/calibration.py`). Out-of-fold train predictions decide whether
  Platt recalibration is needed (decile ECE above 0.005 or slope outside 1 ± 0.1).
* **Master scale** (`core/master_scale.py`). PD maps to grades 1–10.
* **Origination proxy.** An application-leg-only model of the champion's type, used later by
  the ECL layer's relative SICR test (no true origination PD exists in this data).
* **Explainability** (node `explain`, `core/explain.py`): exact TreeSHAP for every OOT applicant,
  reason codes grouped into families, driver stability and a sweep test of the constraints.

In [34]:
import acr.core.metrics
show(acr.core.metrics)

"""Performance, stability and calibration metrics shared by modelling and validation."""

from __future__ import annotations

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import roc_auc_score, roc_curve

EPS = 1e-6


def gini(y, p) -> float:
    return float(2 * roc_auc_score(y, p) - 1)


def ks(y, p) -> float:
    fpr, tpr, _ = roc_curve(y, p)
    return float(np.max(tpr - fpr))


def psi_from_counts(expected: pd.Series, actual: pd.Series) -> float:
    """Population stability index between two distributions over the same bins."""
    idx = expected.index.union(actual.index)
    e = expected.reindex(idx, fill_value=0).astype(float)
    a = actual.reindex(idx, fill_value=0).astype(float)
    e = (e / e.sum()).clip(lower=EPS)
    a = (a / a.sum()).clip(lower=EPS)
    return float(((a - e) * np.log(a / e)).sum())


def psi(expected_bins: pd.Series, actual_bins: pd.Series) -> float:
    return psi_from_counts(expected_bins.value_counts(dropna=False),
                           actual_bins.value_counts(dropna=False))  # fmt: skip


def decile_table(y, p, q: int = 10) -> pd.DataFrame:
    df = pd.DataFrame({"y": np.asarray(y), "p": np.asarray(p)})
    df["bucket"] = pd.qcut(df["p"].rank(method="first"), q, labels=False)
    return df.groupby("bucket").agg(n=("y", "size"), defaults=("y", "sum"), pd_mean=("p", "mean"),
                                    dr=("y", "mean")).reset_index()  # fmt: skip


def ece(y, p, q: int = 10) -> float:
    """Expected calibration error over PD deciles (count-weighted |mean PD - default rate|)."""
    t = decile_table(y, p, q)
    return float((t["n"] * (t["pd_mean"] - t["dr"]).abs()).sum() / t["n"].sum())


def calibration_slope_intercept(y, p) -> tuple[float, float]:
    """Logistic recalibration: logit(P(y=1)) = a + b * logit(p). Perfect calibration: a=0, b=1."""
    import statsmodels.api as sm

    p = np.clip(np.asarray(p, dtype="float64"), EPS, 1 - EPS)
    lp = np.log(p / (1 - p))
    if np.std(lp) < 1e-9:  # constant predictions: slope undefined
        return float("nan"), float("nan")
    res = sm.Logit(np.asarray(y), sm.add_constant(lp, has_constant="add")).fit(disp=0)
    return float(res.params[1]), float(res.params[0])


def hosmer_lemeshow(y, p, q: int = 10) -> tuple[float, float]:
    t = decile_table(y, p, q)
    exp = t["n"] * t["pd_mean"]
    obs = t["defaults"]
    stat = float((((obs - exp) ** 2) / (exp * (1 - t["pd_mean"])).clip(lower=EPS)).sum())
    return stat, float(1 - stats.chi2.cdf(stat, q - 2))

In [35]:
import acr.core.scorecard
show(acr.core.scorecard)

"""Champion: WoE logistic regression scorecard.

Points scaling (standard PDO form):
    factor = PDO / ln(2),   offset = base_score - factor * ln(base_odds)
    score  = offset + factor * ln(good:bad odds) = offset - factor * (b0 + sum_i b_i * WoE_i)
Points for bin j of feature i, with the intercept and offset spread evenly over n features:
    points_ij = -factor * b_i * WoE_ij + (offset - factor * b0) / n
A higher score means lower risk. PD comes from the logistic model, not from rounded points.
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd

from acr.config import ScorecardCfg
from acr.core.selection import logit_fit


@dataclass
class Scorecard:
    features: list[str]
    intercept: float
    coefs: dict[str, float]
    coef_table: pd.DataFrame
    pdo: float
    base_score: float
    base_odds: float

    @property
    def factor(self) -> float:
        return self.pdo / np.log(2)

    @property
    def offset(self) -> float:
        return self.base_score - self.factor * np.log(self.base_odds)

    def linear(self, W: pd.DataFrame) -> np.ndarray:
        b = np.array([self.coefs[f] for f in self.features])
        return self.intercept + W[self.features].to_numpy(dtype="float64") @ b

    def predict_pd(self, W: pd.DataFrame) -> np.ndarray:
        return 1 / (1 + np.exp(-self.linear(W)))

    def score(self, W: pd.DataFrame) -> np.ndarray:
        return self.offset - self.factor * self.linear(W)

    def points_table(self, bin_tables: pd.DataFrame) -> pd.DataFrame:
        n = len(self.features)
        t = bin_tables[bin_tables["feature"].isin(self.features)].copy()
        t = t[t["count"] > 0]
        t["coef"] = t["feature"].map(self.coefs)
        t["points"] = (
            -self.factor * t["coef"] * t["woe"] + (self.offset - self.factor * self.intercept) / n
        )
        t["points_rounded"] = t["points"].round().astype(int)
        order = {f: i for i, f in enumerate(self.features)}
        return t.sort_values(
            ["feature", "bin_index"], key=lambda s: s.map(order) if s.name == "feature" else s
        )


def fit_scorecard(W_train: pd.DataFrame, y_train: pd.Series, features: list[str],
                  cfg: ScorecardCfg) -> Scorecard:  # fmt: skip
    res = logit_fit(W_train[features], y_train)
    names = ["intercept", *features]
    table = pd.DataFrame({"term": names, "coef": res.params, "std_err": res.bse, "z": res.tvalues,
                          "p_value": res.pvalues})  # fmt: skip
    return Scorecard(features, float(res.params[0]), dict(zip(features, map(float, res.params[1:]),
                     strict=True)), table, cfg.pdo, cfg.base_score, cfg.base_odds)  # fmt: skip

In [36]:
import acr.core.challenger
show(acr.core.challenger)

"""Challenger: LightGBM on all candidate features (raw values, not WoE).

Hyperparameters are fixed in config; nothing is tuned automatically, by an LLM or otherwise.
Early stopping uses a stratified slice carved out of train, never test or OOT.
"""

from __future__ import annotations

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from acr.config import LGBMFitCfg


def prepare(X: pd.DataFrame, categorical: list[str]) -> pd.DataFrame:
    X = X.copy()
    for c in X.columns:
        X[c] = X[c].astype("category") if c in categorical else X[c].astype("float32")
    return X


def monotone_signs(bp, features: list[str], categorical: list[str]) -> dict[str, int]:
    """Direction each feature may move PD, from its train-only WoE binning trend.

    +1: higher value can only raise PD; -1: can only lower it; 0: unconstrained (categorical,
    single-bin, or a non-monotone event-rate pattern). Missing and special bins are ignored."""
    signs = {}
    for f in features:
        if f in categorical:
            signs[f] = 0
            continue
        bt = bp.get_binned_variable(f).binning_table.build().drop(index="Totals")
        bt = bt[(bt["Count"] > 0) & ~bt["Bin"].astype(str).isin(["Special", "Missing"])]
        er = bt["Event rate"].to_numpy()
        if len(er) < 2:
            signs[f] = 0
        elif np.all(np.diff(er) >= 0):
            signs[f] = 1
        elif np.all(np.diff(er) <= 0):
            signs[f] = -1
        else:
            signs[f] = 0
    return signs


def constrained(
    params: dict, columns: list[str], signs: dict[str, int] | None, method: str
) -> dict:
    if not signs:
        return params
    return {**params, "monotone_constraints": [signs.get(c, 0) for c in columns],
            "monotone_constraints_method": method}  # fmt: skip


def _params(params: dict, seed: int) -> dict:
    return {**params, "random_state": seed, "deterministic": True, "force_row_wise": True,
            "n_jobs": 4, "verbose": -1}  # fmt: skip


def fit_challenger(X: pd.DataFrame, y: pd.Series, categorical: list[str], params: dict,
                   fit_cfg: LGBMFitCfg, seed: int,
                   signs: dict[str, int] | None = None) -> lgb.LGBMClassifier:  # fmt: skip
    X = prepare(X, categorical)
    params = constrained(params, list(X.columns), signs, fit_cfg.monotone_method)
    Xa, Xv, ya, yv = train_test_split(X, y, test_size=fit_cfg.validation_frac, stratify=y,
                                      random_state=seed)  # fmt: skip
    probe = lgb.LGBMClassifier(**_params(params, seed))
    probe.fit(Xa, ya, eval_X=Xv, eval_y=yv, eval_metric="auc",
              callbacks=[lgb.early_stopping(fit_cfg.early_stopping_rounds, verbose=False)])  # fmt: skip
    best = int(probe.best_iteration_ or params["n_estimators"])
    # Refit on all of train with the early-stopped number of trees.
    final = lgb.LGBMClassifier(**_params({**params, "n_estimators": best}, seed))
    final.fit(X, y)
    return final


def predict_pd(model: lgb.LGBMClassifier, X: pd.DataFrame, categorical: list[str]) -> np.ndarray:
    return model.predict_proba(prepare(X, categorical))[:, 1]


def importance(model: lgb.LGBMClassifier) -> pd.DataFrame:
    gain = model.booster_.feature_importance(importance_type="gain")
    return (pd.DataFrame({"feature": model.booster_.feature_name(), "gain": gain})
            .assign(share=lambda d: d["gain"] / d["gain"].sum())
            .sort_values("gain", ascending=False).reset_index(drop=True))  # fmt: skip

In [37]:
import acr.core.calibration
show(acr.core.calibration)

"""Calibration decision and Platt recalibration, driven by out-of-fold predictions on train.

Hosmer-Lemeshow is reported in validation, but with ~170k observations it rejects almost any
model, so it does not drive the decision here. The decision uses the decile ECE and the
logistic calibration slope of out-of-fold train predictions, with thresholds in config.
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np

from acr.config import CalibrationCfg
from acr.core.metrics import EPS, calibration_slope_intercept, ece


def _logit(p):
    p = np.clip(p, EPS, 1 - EPS)
    return np.log(p / (1 - p))


@dataclass
class Calibrator:
    applied: bool
    slope: float = 1.0
    intercept: float = 0.0
    reason: str = ""
    oof_ece: float = 0.0
    oof_slope: float = 1.0
    oof_intercept: float = 0.0

    def apply(self, p: np.ndarray) -> np.ndarray:
        if not self.applied:
            return np.asarray(p, dtype="float64")
        return 1 / (1 + np.exp(-(self.intercept + self.slope * _logit(p))))


def decide(y_oof, p_oof, cfg: CalibrationCfg) -> Calibrator:
    e = ece(y_oof, p_oof)
    slope, intercept = calibration_slope_intercept(y_oof, p_oof)
    needs = e > cfg.ece_max or abs(slope - 1) > cfg.slope_tol
    reason = (f"OOF ECE {e:.4f} (max {cfg.ece_max}), slope {slope:.3f} (tolerance "
              f"{cfg.slope_tol})")  # fmt: skip
    if not needs:
        return Calibrator(False, reason="not needed: " + reason, oof_ece=e, oof_slope=slope,
                          oof_intercept=intercept)  # fmt: skip
    return Calibrator(True, slope, intercept, "Platt recalibration applied: " + reason, e, slope,
                      intercept)  # fmt: skip

In [38]:
import acr.core.master_scale
show(acr.core.master_scale)

"""Master rating scale: maps PD to grades 1 (best) .. N (worst) using upper PD bounds."""

from __future__ import annotations

import numpy as np
import pandas as pd


def assign_grade(pd_: np.ndarray, upper_bounds: list[float]) -> np.ndarray:
    """Grade k holds PDs in (bound_{k-1}, bound_k]."""
    return np.searchsorted(np.asarray(upper_bounds), np.asarray(pd_), side="left") + 1


def scale_table(upper_bounds: list[float]) -> pd.DataFrame:
    lower = [0.0, *upper_bounds[:-1]]
    return pd.DataFrame({
        "grade": np.arange(1, len(upper_bounds) + 1),
        "pd_lower": lower,
        "pd_upper": upper_bounds,
        "pd_mid": [np.sqrt(max(lo, 1e-4) * hi) if hi < 1 else lo * 1.5
                   for lo, hi in zip(lower, upper_bounds, strict=True)],
    })  # fmt: skip

In [39]:
import acr.nodes.model
show(acr.nodes.model)

"""Node 5: modelling. WoE logistic scorecard and monotone LightGBM, calibration, master scale.

Both models are always built. ``model.champion`` in config assigns the roles: the champion
drives gates, ECL, documents and reason codes; the other model is the benchmark challenger and
documented fallback (decision D33). Scored columns ``pd_champion`` / ``pd_challenger`` follow
the roles; ``pd_scorecard`` / ``pd_lightgbm`` always name the models.

Also fits the application-leg-only "origination proxy" in the champion's model family, used by
the ECL layer's relative SICR test (decision D6). All fitting uses train only.
"""

from __future__ import annotations

import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

from acr.audit import node
from acr.core import binning as B
from acr.core import challenger as C
from acr.core.calibration import decide
from acr.core.guards import require_partition
from acr.core.master_scale import assign_grade, scale_table
from acr.core.metrics import gini, ks
from acr.core.scorecard import fit_scorecard
from acr.core.selection import prune
from acr.nodes._util import artefact_dir, cfg, sha256


def _oof(fit_predict, X: pd.DataFrame, y: pd.Series, folds: int, seed: int) -> np.ndarray:
    oof = np.zeros(len(y))
    for tr, va in StratifiedKFold(folds, shuffle=True, random_state=seed).split(X, y):
        oof[va] = fit_predict(X.iloc[tr], y.iloc[tr], X.iloc[va])
    return oof


@node(
    "model",
    role="Fit the WoE logistic scorecard and monotone LightGBM on train; assign champion and "
    "challenger roles from config; decide calibration from out-of-fold train predictions; map PD "
    "to the master scale; fit an application-only origination proxy in the champion's family.",
    reads=["config", "split", "selection"],
    writes=["model"],
)
def model(view):
    c = cfg(view)
    m, seed = c.base.model, c.base.seed
    sp = view["selection"]["paths"]
    meta = json.loads(Path(sp["meta"]).read_text())
    with open(sp["binning"], "rb") as f:
        bp = pickle.load(f)
    features = pd.read_parquet(sp["features_all"])
    dictionary = pd.read_csv(sp["dictionary"]).set_index("name")
    # Labelled rows come from the split assignment. Any other rows in `features` are the
    # unlabelled application_test applicants: scored, never fitted on.
    a = pd.read_parquet(view["split"]["assignment_path"]).set_index("SK_ID_CURR")
    train_ids = a.index[a["partition"] == "train"]
    y = a["TARGET"]
    ytr = y.loc[train_ids]

    live = list(bp.variable_names)
    cat_live = [f for f in live if f in meta["categorical"]]
    W = B.woe(bp, features[live], cat_live)
    Wtr = require_partition(W.loc[train_ids], a.reset_index(), "train")

    # --- WoE logistic scorecard ---
    sel = meta["selected"]
    sc = fit_scorecard(Wtr, ytr, sel, m.scorecard)

    def lr_fp(Xa, ya, Xb):
        return fit_scorecard(Xa, ya, sel, m.scorecard).predict_pd(Xb)

    oof_ch = _oof(lr_fp, Wtr, ytr, m.calibration.cv_folds, seed)
    cal_ch = decide(ytr, oof_ch, m.calibration)
    pd_ch_raw = sc.predict_pd(W)
    pd_ch = cal_ch.apply(pd_ch_raw)

    # --- monotone LightGBM ---
    excluded = set(c.base.features.exclude_features)
    cands = [f for f in meta["candidates"] if f in live and f not in excluded]
    Xtr = require_partition(features.loc[train_ids, cands], a.reset_index(), "train")
    # Monotonic constraints: directions from the train-only WoE binning trends.
    signs = C.monotone_signs(bp, cands, cat_live) if m.lightgbm_fit.monotone else None
    lgbm = C.fit_challenger(Xtr, ytr, cat_live, dict(m.lightgbm), m.lightgbm_fit, seed, signs)
    n_trees = int(lgbm.n_estimators)
    fixed = C.constrained({**dict(m.lightgbm), "n_estimators": n_trees}, cands, signs,
                          m.lightgbm_fit.monotone_method)  # fmt: skip

    def gb_fp(Xa, ya, Xb):
        import lightgbm as lgb

        mdl = lgb.LGBMClassifier(**C._para

In [40]:
from acr.nodes.model import model

step(model, state)
m = state["model"]
display(pd.DataFrame({k: {kk: m[k].get(kk) for kk in ("type", "n_features", "train_auc", "train_gini", "oof_auc", "oof_gini", "calibration")}
                      for k in ("champion", "challenger")}).T)
print("roles:", m["roles"])
print("origination proxy features:", m["origination_proxy"]["features"])

model: wrote ['model']


,type,n_features,train_auc,train_gini,oof_auc,oof_gini,calibration
champion,monotone LightGBM gradient boosting,112,0.8456,0.6911,0.77,0.5401,Platt recalibration applied: OOF ECE 0.0069 (m...
challenger,WoE logistic regression scorecard,10,0.7685,0.537,0.7668,0.5336,"not needed: OOF ECE 0.0041 (max 0.005), slope ..."


roles: {'champion': 'lightgbm', 'challenger': 'scorecard'}
origination proxy features: ['APP_EXT_SOURCE_1', 'APP_EXT_SOURCE_2', 'APP_EXT_SOURCE_3', 'APP_AMT_INCOME_TOTAL', 'APP_AMT_CREDIT', 'APP_AMT_ANNUITY', 'APP_AMT_GOODS_PRICE', 'APP_CNT_CHILDREN', 'APP_CNT_FAM_MEMBERS', 'APP_REGION_POPULATION_RELATIVE', 'APP_REGION_RATING_CLIENT_W_CITY', 'APP_OWN_CAR_AGE', 'APP_AMT_REQ_CREDIT_BUREAU_MON', 'APP_AMT_REQ_CREDIT_BUREAU_QRT', 'APP_AMT_REQ_CREDIT_BUREAU_YEAR', 'APP_DEF_30_CNT_SOCIAL_CIRCLE', 'APP_DEF_60_CNT_SOCIAL_CIRCLE', 'APP_OBS_30_CNT_SOCIAL_CIRCLE', 'APP_FLAG_DOCUMENT_3', 'APP_REG_CITY_NOT_WORK_CITY', 'APP_REG_CITY_NOT_LIVE_CITY', 'APP_FLAG_WORK_PHONE', 'APP_FLAG_PHONE', 'APP_TOTALAREA_MODE', 'APP_AGE_YEARS', 'APP_EMPLOYED_YEARS', 'APP_EMPLOYED_SENTINEL_FLAG', 'APP_EMPLOYED_TO_AGE', 'APP_REGISTRATION_YEARS', 'APP_ID_PUBLISH_YEARS', 'APP_PHONE_CHANGE_YEARS', 'APP_CREDIT_TO_INCOME', 'APP_ANNUITY_TO_INCOME', 'APP_CREDIT_TO_GOODS', 'APP_CREDIT_TO_ANNUITY', 'APP_INCOME_PER_PERSON', 'APP_

In [41]:
# Scorecard (benchmark) coefficient table: every WoE coefficient should be negative
display(pd.read_csv(m["paths"]["coefficients"]).round(4))

,term,coef,std_err,z,p_value
0,intercept,-2.4322,0.0313,-77.6215,0.0000
1,APP_EXT_SOURCE_3,-0.7501,0.0500,-15.0102,0.0000
2,APP_EXT_SOURCE_2,-0.8041,0.0514,-15.6538,0.0000
3,APP_EXT_SOURCE_1,-0.6140,0.0778,-7.8924,0.0000
4,APP_OCCUPATION_TYPE,-0.5556,0.0896,-6.1998,0.0000
5,BUR_MONTHS_SINCE_FIRST,-0.3090,0.0926,-3.3385,0.0008
6,BEH_CC_UTIL_MEAN,-0.5705,0.0870,-6.5540,0.0000
7,BEH_INS_DPD_MAX_12M,-0.5914,0.0961,-6.1529,0.0000
8,APP_ORGANIZATION_TYPE,-0.5471,0.1114,-4.9120,0.0000
9,BUR_ACTIVE_DEBT_TO_CREDIT,-0.5601,0.1048,-5.3449,0.0000


In [42]:
# Scorecard points for one characteristic (bins, WoE, points). Higher points = lower risk.
pts = pd.read_csv(m["paths"]["points"])
first = pts["feature"].iloc[0]
display(pts[pts["feature"] == first][["feature", "bin", "count", "event_rate", "woe", "points_rounded"]])

,feature,bin,count,event_rate,woe,points_rounded
0,APP_EXT_SOURCE_3,"(-inf, 0.18)",904,0.222345,-1.180719,30
1,APP_EXT_SOURCE_3,"[0.18, 0.30)",1407,0.160625,-0.779189,39
2,APP_EXT_SOURCE_3,"[0.30, 0.47)",3216,0.087687,-0.090556,54
3,APP_EXT_SOURCE_3,"[0.47, 0.62)",3457,0.061325,0.295513,62
4,APP_EXT_SOURCE_3,"[0.62, 0.78)",4020,0.035821,0.859975,74
5,APP_EXT_SOURCE_3,"[0.78, inf)",878,0.022779,1.326101,84
6,APP_EXT_SOURCE_3,Missing,3415,0.091069,-0.132117,53


In [43]:
import acr.core.explain
show(acr.core.explain)

"""Explainability for the LightGBM challenger: exact TreeSHAP and checks on it.

SHAP values come from LightGBM itself (``pred_contrib=True``): an exact, deterministic
TreeSHAP decomposition in log-odds. For every applicant, bias + sum of feature
contributions = the model's raw score, so each prediction is fully attributed.

Caveats stated in the outputs: SHAP describes how the model attributes a prediction, not
cause and effect; correlated features (e.g. EXT_SOURCE_MEAN vs EXT_SOURCE_1/2/3) can share
credit arbitrarily, which is why importance is also reported by leg.
"""

from __future__ import annotations

import lightgbm as lgb
import numpy as np
import pandas as pd
from scipy.stats import spearmanr


def shap_frame(booster: lgb.Booster, X: pd.DataFrame) -> pd.DataFrame:
    contrib = booster.predict(X, pred_contrib=True)
    return pd.DataFrame(contrib, index=X.index, columns=[*booster.feature_name(), "bias"])


def additivity_error(booster: lgb.Booster, X: pd.DataFrame, sh: pd.DataFrame) -> float:
    return float(np.abs(sh.sum(axis=1).to_numpy() - booster.predict(X, raw_score=True)).max())


def global_importance(sh: pd.DataFrame) -> pd.DataFrame:
    f = sh.drop(columns="bias")
    imp = pd.DataFrame({"mean_abs_shap": f.abs().mean(), "mean_shap": f.mean()})
    total = imp["mean_abs_shap"].sum()
    imp["share"] = (
        imp["mean_abs_shap"] / total if total > 0 else 0.0
    )  # all-zero: model made no splits
    return imp.sort_values("mean_abs_shap", ascending=False)


def leg_importance(imp: pd.DataFrame, leg_of: pd.Series) -> pd.Series:
    return imp.groupby(leg_of.reindex(imp.index))["share"].sum().sort_values(ascending=False)


def reason_codes(sh: pd.DataFrame, X: pd.DataFrame, k: int) -> pd.DataFrame:
    """Top-k features that pushed each applicant's PD *up* (largest positive SHAP), with the
    applicant's value for that feature. Long format: one row per applicant and rank."""
    f = sh.drop(columns="bias")
    vals = f.to_numpy()
    order = np.argsort(-vals, axis=1)[:, :k]
    cols = np.asarray(f.columns)
    rows = []
    for i, idx in enumerate(f.index):
        for rank, j in enumerate(order[i], 1):
            if vals[i, j] <= 0:
                break
            v = X.iat[i, X.columns.get_loc(cols[j])]
            rows.append((idx, rank, cols[j], v if pd.notna(v) else None, float(vals[i, j])))
    out = pd.DataFrame(rows, columns=["SK_ID_CURR", "rank", "feature", "value", "shap_logodds"])
    out["value"] = out["value"].astype("string")
    return out


def stability(imp_a: pd.DataFrame, imp_b: pd.DataFrame, top_n: int) -> dict:
    """Do two samples rank the same drivers? Top-N overlap and Spearman over all features."""
    a, b = imp_a["mean_abs_shap"], imp_b["mean_abs_shap"].reindex(imp_a.index)
    top_a, top_b = set(a.nlargest(top_n).index), set(b.nlargest(top_n).index)
    if a.sum() == 0 or b.sum() == 0:  # a model with no splits explains nothing
        return {"top_n": top_n, "top_overlap": 0.0, "spearman": float("nan")}
    return {"top_n": top_n, "top_overlap": len(top_a & top_b) / top_n,
            "spearman": float(spearmanr(a, b).statistic)}  # fmt: skip


def monotonicity_violations(booster: lgb.Booster, X: pd.DataFrame, signs: dict[str, int],
                            grid: int = 11) -> pd.DataFrame:  # fmt: skip
    """Sweep each constrained feature across its quantiles for every row of X and count rows
    whose prediction moves against the allowed direction (tolerance 1e-9 in log-odds)."""
    rows = []
    for f, s in signs.items():
        if s == 0 or f not in X.columns:
            continue
        qs = np.unique(np.nanquantile(X[f].astype(float), np.linspace(0, 1, grid)))
        if len(qs) < 2:
            continue
        preds = []
        for q in qs:
            Xq = X.copy()
            Xq[f] = np.float32(q)
            preds.append(booster.predict(Xq, raw_score=True))
        steps = np.diff(np.vstack(preds), axis=0) * s
        rows.append({"feature": f, "sign":

In [44]:
from acr.nodes.explain import explain

step(explain, state)
X = state["explain"]
print("applicants explained:", X["applicants_explained"], "| constraint violations:", X["monotone"]["violating_rows"])
display(pd.Series(X["family_shares"]).rename("share of SHAP weight").to_frame().style.format("{:.1%}"))

explain: wrote ['explain']
applicants explained: 7688 | constraint violations: 0


,share of SHAP weight
External credit scores,29.4%
Repayment history on previous Home Credit loans,11.2%
Employment and occupation,10.5%
Credit with other lenders (bureau),9.8%
Previous applications and refusals,9.8%
Affordability (instalment and loan size against income),9.4%
Applicant profile and contact details,7.9%
Credit card usage,7.4%
Loan amount and terms,4.6%


## 10. Independent validation (node `validate`): read-only

The second-line validator works only on **scored outputs**. It imports no fitting code, and it
checks every model artefact's SHA-256 against the checksums the model node recorded,
**before and after** it runs. A mismatch is a red finding.

What it computes:
* Gini/AUC, KS and Brier by partition
* score PSI and per-feature CSI, plus a stress-PSI ladder
* calibration: HL, ECE, slope, and a one-sided binomial traffic light by grade
* rank-ordering
* OOT segments, including gender as a fairness check
* champion vs challenger, with a bootstrap 95% CI
* an external score-PSI and CSI check on the unlabelled `application_test` population

Every Gini is reported with its **AUC** (Gini = 2 × AUC − 1), the metric Kaggle uses.

`core/findings.py` turns these into **structured findings** with severities (info → red),
each stating its threshold. Only **red** findings drive the gate: they halt full runs and
only warn on `--sample`.

In [45]:
import acr.core.validation
show(acr.core.validation)

"""Independent validation metrics. Pure functions over *scored outputs*; nothing here fits,
tunes or alters a model.

Pseudo-OOT note: train, test and OOT are random stratified draws from the same population,
so PSI and CSI are close to zero by construction. They test the plumbing, not real drift.
``stress_psi`` shows what the same metric reports under a deliberate population shift.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import brier_score_loss, roc_auc_score

from acr.core.metrics import calibration_slope_intercept, ece, gini, hosmer_lemeshow, ks, psi

PARTS = ("train", "test", "oot")


def perf(y: pd.Series, p: pd.Series) -> dict:
    auc = roc_auc_score(y, p)
    return {"n": int(len(y)), "defaults": int(y.sum()), "default_rate": float(y.mean()),
            "mean_pd": float(np.mean(p)), "auc": float(auc), "gini": float(2 * auc - 1),
            "ks": ks(y, p), "brier": float(brier_score_loss(y, p))}  # fmt: skip


def by_partition(scored: pd.DataFrame, pd_col: str) -> dict:
    return {
        part: perf(g["TARGET"], g[pd_col])
        for part, g in scored.groupby("partition")
        if part in PARTS
    }


def calibration(y: pd.Series, p: pd.Series) -> dict:
    hl, hl_p = hosmer_lemeshow(y, p)
    slope, intercept = calibration_slope_intercept(y, p)
    return {"hl_stat": hl, "hl_p_value": hl_p, "ece": ece(y, p), "slope": slope,
            "intercept": intercept, "pd_to_dr": float(np.mean(p) / max(y.mean(), 1e-12))}  # fmt: skip


def _decile_edges(train_score: pd.Series) -> np.ndarray:
    edges = np.unique(np.quantile(train_score, np.linspace(0, 1, 11)))
    edges[0], edges[-1] = -np.inf, np.inf
    return edges


def score_psi(scored: pd.DataFrame, col: str = "pd_champion") -> dict:
    """PSI of the champion's PD distribution over train-defined deciles (deciles of PD and of a
    monotone score are the same, so this equals score PSI)."""
    edges = _decile_edges(scored.loc[scored["partition"] == "train", col])
    b = pd.cut(scored[col], edges, labels=False)
    base = b[scored["partition"] == "train"]
    return {f"train_vs_{p}": psi(base, b[scored["partition"] == p]) for p in ("test", "oot")}


def grade_psi(scored: pd.DataFrame, grade_col: str) -> dict:
    base = scored.loc[scored["partition"] == "train", grade_col]
    return {f"train_vs_{p}": psi(base, scored.loc[scored["partition"] == p, grade_col])
            for p in ("test", "oot")}  # fmt: skip


def stress_psi(scored: pd.DataFrame, age_years: pd.Series, young_weight: float,
               col: str = "pd_champion") -> float:  # fmt: skip
    """Illustrative drift: weight OOT applicants under 30 by ``young_weight`` and recompute
    the score PSI against train. Shows the PSI gate firing on a shifted book."""
    edges = _decile_edges(scored.loc[scored["partition"] == "train", col])
    b = pd.cut(scored[col], edges, labels=False)
    base = b[scored["partition"] == "train"].value_counts()
    oot = scored["partition"] == "oot"
    w = np.where(age_years.reindex(scored.index)[oot] < 30, young_weight, 1.0)
    stressed = pd.Series(w, index=b[oot].index).groupby(b[oot]).sum()
    from acr.core.metrics import psi_from_counts

    return psi_from_counts(base, stressed)


def grade_table(df: pd.DataFrame, pd_col: str, grade_col: str, alpha_red: float,
                alpha_amber: float) -> pd.DataFrame:  # fmt: skip
    """Per-grade outcomes with one-sided binomial tests.

    ``p_under``: probability of at least the observed defaults if the true PD were the
    grade's mean PD. Small values mean the model under-predicts risk, which is the
    prudentially relevant direction. Traffic light: red at ``alpha_red``, amber at
    ``alpha_amber``. The test assumes independent defaults, so it is conservative in a
    real (correlated) portfolio.
    """
    g = df.groupby(grade_col).agg(n=("TARGET", "size"), defaults=("TARGET", "sum"),
                                  mean_pd=(

In [46]:
import acr.core.findings
show(acr.core.findings)

"""Deterministic validation findings: rules over validation metrics -> structured findings.

Each rule states its threshold (from config/gates.yaml) in the finding text, so a reader can
see why a severity was assigned. The LLM critic (Phase 6) can add challenges on top of these,
but cannot change or remove them.
"""

from __future__ import annotations

import pandas as pd

from acr.config import GateSet, ValidationCfg
from acr.state import Finding


def build_findings(m: dict, grades: pd.DataFrame, csi: pd.DataFrame, seg: pd.DataFrame,
                   v: ValidationCfg, gate: GateSet, is_sample: bool) -> list[Finding]:  # fmt: skip
    out: list[Finding] = []

    def add(area, severity, title, detail, remediation=""):
        out.append(Finding(id=f"V{len(out) + 1:02d}", area=area, severity=severity, title=title,
                           detail=detail, remediation=remediation))  # fmt: skip

    if not m["integrity"]["ok"]:
        add(
            "Integrity",
            "red",
            "Model artefacts changed after modelling",
            f"Checksum mismatch: {m['integrity']['mismatches']}. Validation results cannot be relied on.",
            "Re-run modelling; investigate any process that writes to model artefacts.",
        )

    ch = m["champion"]["performance"]
    for part, floor in (("test", gate.gini_min_test), ("oot", gate.gini_min_oot)):
        g = ch[part]["gini"]
        if g < floor:
            add(
                "Performance",
                "red",
                f"Champion {part} Gini below gate",
                f"{part} Gini {g:.4f} (AUC {ch[part]['auc']:.4f}) is below the minimum {floor:.2f}.",
                "Revisit feature engineering and selection; do not deploy.",
            )
        elif g < floor + v.gini_amber_margin:
            add(
                "Performance",
                "medium",
                f"Champion {part} Gini close to gate",
                f"{part} Gini {g:.4f} (AUC {ch[part]['auc']:.4f}) is within {v.gini_amber_margin:.2f} "
                f"of the {floor:.2f} minimum.",
                "Monitor closely; set an early-warning trigger in the monitoring plan.",
            )

    roles = m.get("roles", {"champion": "champion", "challenger": "challenger"})
    names = {"lightgbm": "monotone LightGBM", "scorecard": "WoE scorecard"}
    ch_name = names.get(roles["champion"], "champion")
    cl_name = names.get(roles["challenger"], "challenger")
    cl = m["challenger"]["performance"]
    for role, name, p in (("Champion", ch_name, ch), ("Challenger", cl_name, cl)):
        gap = p["train"]["gini"] - p["oot"]["gini"]
        if gap > v.overfit_gap_medium:
            add(
                "Performance" if role == "Champion" else "Benchmarking",
                "medium",
                f"{role} ({name}) overfits the training data",
                f"Train Gini {p['train']['gini']:.4f} (AUC {p['train']['auc']:.4f}) vs OOT Gini "
                f"{p['oot']['gini']:.4f} (AUC {p['oot']['auc']:.4f}); gap {gap:.4f}, threshold "
                f"{v.overfit_gap_medium:.2f}.",
                "Simplify the model or strengthen regularisation; compare on holdout only.",
            )

    cmp_ = m["comparison"]  # diff = challenger Gini - champion Gini on OOT
    detail = (f"Champion ({ch_name}) OOT Gini {ch['oot']['gini']:.4f} (AUC {ch['oot']['auc']:.4f}) vs "
              f"challenger ({cl_name}) {cl['oot']['gini']:.4f} (AUC {cl['oot']['auc']:.4f}); "
              f"challenger minus champion {cmp_['diff']:.4f} (95% bootstrap CI {cmp_['ci_low']:.4f} "
              f"to {cmp_['ci_high']:.4f}; AUC difference {cmp_['diff'] / 2:.4f}).")  # fmt: skip
    if cmp_["diff"] > v.challenger_gap_medium:
        add("Benchmarking", "medium", "Challenger materially outperforms champion",
            detail + f" The gap exceeds {v.challenger_gap_medium:.2f}.",
            "Document why the champion is preferred, or review the champion choice.")  # fmt: skip
    elif cmp_["ci_low"] > 0

In [47]:
import acr.nodes.validate
show(acr.nodes.validate)

"""Node 6: independent validation (read-only over model artefacts) and the performance gate.

Second-line controls:
* Integrity: every model artefact's SHA-256 is checked against the checksum recorded by the
  model node, before and after validation. A mismatch is a red finding.
* Independence: this module imports no fitting code. It reads scored outputs, the points
  table and the train-fitted binning (transform only).
"""

from __future__ import annotations

import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

from acr.audit import node
from acr.core import binning as B
from acr.core import validation as V
from acr.core.findings import build_findings, gate_decision
from acr.nodes._util import artefact_dir, cfg, is_sample, sha256

EXTERNAL = ("APP_EXT_SOURCE_1", "APP_EXT_SOURCE_2", "APP_EXT_SOURCE_3", "APP_EXT_SOURCE_MEAN",
            "APP_EXT_SOURCE_MIN")  # fmt: skip


def _integrity(model: dict) -> dict:
    bad = [k for k, p in model["paths"].items() if sha256(p) != model["checksums"][k]]
    return {"ok": not bad, "mismatches": bad}


def _r(x):
    """Round floats for state and reports (6 dp) so numbers are stable and citable."""
    if isinstance(x, dict):
        return {k: _r(v) for k, v in x.items()}
    if isinstance(x, (float, np.floating)):
        return round(float(x), 6)
    if isinstance(x, (np.integer,)):
        return int(x)
    return x


def _segment_frame(features: pd.DataFrame) -> pd.DataFrame:
    age = features["APP_AGE_YEARS"]
    return pd.DataFrame({
        "contract_type": features["APP_NAME_CONTRACT_TYPE"],
        "income_type": features["APP_NAME_INCOME_TYPE"],
        "gender": features["APP_CODE_GENDER"],
        "age_band": pd.cut(age, [0, 30, 45, 60, 200], right=False,
                           labels=["<30", "30-44", "45-59", "60+"]),
        "bureau_history": np.where(features["BUR_COUNT"] > 0, "yes", "no"),
    }, index=features.index)  # fmt: skip


@node(
    "validate",
    role="Independent second-line validation over scored outputs: Gini/AUC/KS by partition, "
    "score PSI and feature CSI, calibration (HL, ECE, slope, grade-level binomial tests), "
    "rank-ordering, segments, champion vs challenger with bootstrap CI; structured findings; "
    "performance and stability gates. Read-only: artefact checksums verified before and after.",
    reads=["config", "sample_frac", "split", "model", "selection", "explain"],
    writes=["validation", "gate_log"],
    on_failure="A red finding with gate enforcement on routes to the halt node with a halt report.",
)
def validate(view):
    c = cfg(view)
    v, gate, sample = c.gates.validation, c.gate_set(is_sample(view)), is_sample(view)
    model, sel = view["model"], view["selection"]
    integrity_before = _integrity(model)

    scored = pd.read_parquet(model["paths"]["scored"])
    points = pd.read_csv(model["paths"]["points"])
    features = pd.read_parquet(sel["paths"]["features_all"])
    meta = json.loads(Path(sel["paths"]["meta"]).read_text())
    with open(sel["paths"]["binning"], "rb") as f:
        bp = pickle.load(f)
    selected = meta["selected"]
    oot = scored[scored["partition"] == "oot"]

    m: dict = {"champion": {}, "challenger": {}}
    for name, col, gcol in (("champion", "pd_champion", "grade_champion"),
                            ("challenger", "pd_challenger", "grade_challenger")):  # fmt: skip
        m[name]["performance"] = V.by_partition(scored, col)
        m[name]["calibration"] = {p: V.calibration(g["TARGET"], g[col])
                                  for p, g in scored.groupby("partition")}  # fmt: skip
        m[name]["grade_psi"] = V.grade_psi(scored, gcol)
    grades = V.grade_table(oot, "pd_champion", "grade_champion", v.binomial_alpha_red,
                           v.binomial_alpha_amber)  # fmt: skip
    deciles = V.decile_table(oot, "pd_champion")
    m["champion"]["rank_breaks_grade_oot"] = V.rank_order_breaks(
        grades["default_rate"], grades["n"]


In [48]:
from acr.nodes.validate import validate

step(validate, state)
v = state["validation"]
perf = {f"{model_} {part}": x for model_ in ("champion", "challenger")
        for part, x in v["metrics"][model_]["performance"].items()}
display(pd.DataFrame(perf).T[["n", "default_rate", "mean_pd", "auc", "gini", "ks", "brier"]].round(4))
print("gate decision:", v["gate_decision"], "| findings by severity:", v["findings_by_severity"])

validate: wrote ['gate_log', 'validation']


,n,default_rate,mean_pd,auc,gini,ks,brier
champion oot,7688.0,0.0808,0.0810,0.7513,0.5026,0.3886,0.0679
champion test,5766.0,0.0806,0.0802,0.7611,0.5222,0.3971,0.0671
champion train,17297.0,0.0807,0.0814,0.8456,0.6911,0.5341,0.0617
challenger oot,7688.0,0.0808,0.0812,0.7295,0.4591,0.3521,0.0689
challenger test,5766.0,0.0806,0.0806,0.7350,0.4699,0.3458,0.0689
challenger train,17297.0,0.0807,0.0807,0.7685,0.5370,0.4121,0.0678


gate decision: continue | findings by severity: {'red': 0, 'high': 1, 'medium': 3, 'low': 5, 'info': 5}


In [49]:
# Structured findings (sample run: the gates are warn-only)
display(pd.DataFrame(v["findings"])[["id", "severity", "area", "title", "detail"]])

,id,severity,area,title,detail
0,V01,medium,Performance,Champion (monotone LightGBM) overfits the trai...,Train Gini 0.6911 (AUC 0.8456) vs OOT Gini 0.5...
1,V02,info,Benchmarking,Champion outperforms the benchmark,Champion (monotone LightGBM) OOT Gini 0.5026 (...
2,V03,medium,Stability,Stability testing is not temporal (pseudo-OOT),Holdouts are random stratified draws with no a...
3,V04,info,Stability,External population (application_test) score d...,"Score PSI of 4,874 unlabelled application_test..."
4,V05,low,Calibration,Amber calibration in some grades (OOT),1 grade(s) at 0.05 significance: 1.
5,V06,high,Discrimination,Rank-ordering breaks across grades (OOT),1 reversal(s) in default rate across grades.
6,V07,low,Master scale,Sparsely populated grades,Grade(s) 1 hold under 1% of the OOT book (7 ac...
7,V08,low,Segments,Segment calibration off in income_type = Comme...,"Mean PD / default rate 1.106 (tolerance ±10%, ..."
8,V09,low,Segments,Segment calibration off in income_type = Pensi...,"Mean PD / default rate 1.114 (tolerance ±10%, ..."
9,V10,low,Conceptual soundness,Champion uses undisclosed external scores,EXT_SOURCE features carry 22% of mean absolute...


In [50]:
# Grade-level calibration on OOT with binomial traffic lights
display(pd.read_csv(v["paths"]["grade_table_oot"]).round(4))

,grade,n,defaults,mean_pd,share,default_rate,p_under,p_over,traffic_light
0,1,7,1.0,0.0045,0.0009,0.1429,0.0312,0.9996,amber
1,2,181,4.0,0.0081,0.0235,0.0221,0.0597,0.9838,green
2,3,1022,21.0,0.0152,0.1329,0.0205,0.1075,0.9293,green
3,4,1533,46.0,0.0272,0.1994,0.0300,0.2700,0.7778,green
4,5,1061,40.0,0.0421,0.1380,0.0377,0.7818,0.2676,green
5,6,1154,77.0,0.0614,0.1501,0.0667,0.2428,0.7929,green
6,7,774,59.0,0.0871,0.1007,0.0762,0.8743,0.1552,green
7,8,790,95.0,0.1228,0.1028,0.1203,0.6035,0.4390,green
8,9,738,140.0,0.1927,0.0960,0.1897,0.5957,0.4409,green
9,10,428,138.0,0.3308,0.0557,0.3224,0.6607,0.3778,green


In [51]:
# Segment performance on OOT (gender is a fairness check; it is not a model input)
display(pd.read_csv(v["paths"]["segments_oot"]).round(3))

,segment,value,n,default_rate,mean_pd,pd_to_dr,gini,auc
0,contract_type,Cash loans,6966,0.085,0.083,0.978,0.498,0.749
1,contract_type,Revolving loans,722,0.037,0.058,1.543,0.458,0.729
2,income_type,Businessman,1,0.000,0.044,NaN,NaN,NaN
3,income_type,Commercial associate,1788,0.067,0.074,1.106,0.500,0.750
4,income_type,Pensioner,1345,0.052,0.058,1.114,0.351,0.675
5,income_type,State servant,583,0.060,0.069,1.153,0.497,0.749
6,income_type,Student,1,0.000,0.225,NaN,NaN,NaN
7,income_type,Working,3970,0.100,0.094,0.938,0.510,0.755
8,gender,F,5047,0.072,0.075,1.041,0.512,0.756
9,gender,M,2641,0.097,0.092,0.948,0.475,0.737


## 11. Illustrative RBI ECL (node `ecl`)

**Illustrative simulation.** Home Credit is an application-level dataset, not a live book.
The pseudo-OOT partition is treated as a book at a simulated reporting date, with every loan
at origination. Regulatory parameters are in `config/ecl_rbi.yaml` with paragraph
citations to RBI/DOR/2026-27/398 (Directions dated 27 April 2026, effective 1 April 2027).

| Component | Approach | Source |
|---|---|---|
| Stage 3 | more than 90 DPD in the last 12 months on prior Home Credit loans (borrower level) | para 25, NPA definition |
| Stage 2 | more than 30 DPD (Home Credit or bureau), or grade down ≥ 3 notches vs the origination proxy, or PD ≥ 2× proxy PD and ≥ 5% | para 33 presumption; para 26 relative test (proxy, D6) |
| 12-month PD | champion PD, floored at 0.03% | para 96 |
| Lifetime PD | constant hazard (primary) and a bureau DPD-bucket Markov chain scaled to each 12-month PD | D4 |
| Term and EIR | monthly rate implied by prior approved Home Credit loans (train customers); term solved from the annuity formula | D26 |
| EAD | amortising balance; revolving: limit × CCF over a 24-month life | para 46(2)(iii) |
| LGD | 70% unsecured backstop, with a sensitivity ladder | para 97 |
| Scenarios | base / upside / downside, weights and PD scalars (illustrative) | para 48 |
| Floors | Stage 1 1%, Stage 2 5%, Stage 3 25% in the first year | para 82(3)(x) |

The engine works at LGD = 1, so the LGD sensitivity is exact (ECL is linear in LGD).
Implied rates and the transition matrix are learned from **train customers only**.

In [52]:
import acr.core.ecl.rates
show(acr.core.ecl.rates)

"""Annuity maths: implied rates, contractual term and amortising exposure.

Annuity identity for a loan of principal P, monthly instalment A, monthly rate r, n months:
    A = P * r / (1 - (1 + r) ** -n)
Given (P, A, n) the rate solves by bisection (A is increasing in r). Given (P, A, r):
    n = -ln(1 - r * P / A) / ln(1 + r)
"""

from __future__ import annotations

import numpy as np
import pandas as pd

R_MAX = 0.5  # monthly rate search ceiling (600% a year); far above anything in the data


def annuity(P, r, n):
    P, r, n = (np.asarray(x, dtype="float64") for x in (P, r, n))
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(r > 0, P * r / (1 - (1 + r) ** -n), P / n)


def solve_rate(P, A, n, iters: int = 80) -> np.ndarray:
    """Monthly rate implied by principal, instalment and term (vectorised bisection).
    Loans with A * n <= P (no interest) get rate 0."""
    P, A, n = (np.asarray(x, dtype="float64") for x in (P, A, n))
    lo, hi = np.zeros_like(P), np.full_like(P, R_MAX)
    for _ in range(iters):
        mid = (lo + hi) / 2
        too_low = annuity(P, mid, n) < A
        lo, hi = np.where(too_low, mid, lo), np.where(too_low, hi, mid)
    r = (lo + hi) / 2
    return np.where(A * n <= P, 0.0, r)


def implied_rates(prev: pd.DataFrame) -> dict[str, float]:
    """Median implied monthly rate per contract type, from approved prior loans with a known
    term. ``prev`` needs NAME_CONTRACT_TYPE, NAME_CONTRACT_STATUS, AMT_CREDIT, AMT_ANNUITY,
    CNT_PAYMENT."""
    ok = ((prev["NAME_CONTRACT_STATUS"].astype("string") == "Approved")
          & (prev["CNT_PAYMENT"] > 0) & (prev["AMT_ANNUITY"] > 0) & (prev["AMT_CREDIT"] > 0))  # fmt: skip
    p = prev[ok]
    r = solve_rate(p["AMT_CREDIT"], p["AMT_ANNUITY"], p["CNT_PAYMENT"])
    s = pd.Series(r, index=p.index).groupby(p["NAME_CONTRACT_TYPE"].astype("string")).median()
    return {str(k): float(v) for k, v in s.items()}


def term_months(P, A, r, min_months: int, max_months: int) -> np.ndarray:
    """Contractual term solved from the annuity identity at rate r, rounded up and clipped.
    If the instalment does not cover interest (r * P >= A) the term is set to the maximum."""
    P, A, r = (np.asarray(x, dtype="float64") for x in (P, A, r))
    with np.errstate(divide="ignore", invalid="ignore"):
        x = 1 - r * P / A
        n = np.where(r > 0, -np.log(x) / np.log1p(r), P / A)
    n = np.where((x <= 0) | ~np.isfinite(n), max_months, np.ceil(n - 1e-9))
    return np.clip(n, min_months, max_months).astype(int)


def balance_schedule(P, A, r, n, horizon: int) -> np.ndarray:
    """Outstanding balance at the *start* of months 1..horizon (exposure if default occurs in
    that month). Shape (accounts, horizon). Zero after the final instalment."""
    P, A, r = (np.asarray(x, dtype="float64")[:, None] for x in (P, A, r))
    n = np.asarray(n)[:, None]
    t = np.arange(horizon)[None, :]  # months already paid before month t+1
    growth = (1 + r) ** t
    with np.errstate(divide="ignore", invalid="ignore"):
        paid = np.where(r > 0, A * (growth - 1) / r, A * t)
    bal = np.clip(P * growth - paid, 0, None)
    return np.where(t < n, bal, 0.0)

In [53]:
import acr.core.ecl.staging
show(acr.core.ecl.staging)

"""Stage allocation for the simulated reporting-date book (decisions D5, D6).

Stage 3 (credit-impaired, para 25): more than 90 days past due within the lookback on the
  borrower's prior Home Credit loans (same lender, borrower level).
Stage 2 (SICR, paras 26-34): any of
  * rebuttable presumption (para 33): more than 30 DPD within the lookback on prior Home
    Credit loans, or on bureau credits with other lenders (bureau bucket 2+, or currently
    more than 30 days overdue);
  * relative deterioration: reporting grade at least N notches worse than the
    origination-proxy grade; or PD at least X times the proxy PD *and* above an absolute
    floor. The proxy stands in for an origination-date PD, which this data does not have.
Stage 1: everything else.
Every account records the triggers that fired, so the allocation can be audited.
"""

from __future__ import annotations

import numpy as np
import pandas as pd

# Leg features the rules read. All use 12-month windows built in the legs.
INPUTS = ["BEH_INS_DPD_MAX_12M", "BEH_POS_DPD_MAX_12M", "BUR_BB_WORST_BUCKET_12M",
          "BUR_DAYS_OVERDUE_MAX"]  # fmt: skip
FEATURE_WINDOW_MONTHS = 12


def allocate(df: pd.DataFrame, s: dict) -> pd.DataFrame:
    """``df`` holds the INPUTS plus grade_champion, grade_origination_proxy, pd_champion and
    pd_origination_proxy. ``s`` is the staging config. Returns stage and trigger columns."""
    if int(s["lookback_months"]) != FEATURE_WINDOW_MONTHS:
        raise ValueError("staging lookback must match the 12-month leg feature windows")
    hc_dpd = df[["BEH_INS_DPD_MAX_12M", "BEH_POS_DPD_MAX_12M"]].max(axis=1).fillna(0)
    t = pd.DataFrame(index=df.index)
    t["hc_90dpd"] = hc_dpd > s["credit_impaired_dpd"]
    t["hc_30dpd"] = hc_dpd > s["sicr_dpd_presumption"]
    t["bureau_30dpd"] = (df["BUR_BB_WORST_BUCKET_12M"].fillna(0) >= 2) | (
        df["BUR_DAYS_OVERDUE_MAX"].fillna(0) > s["sicr_dpd_presumption"])  # fmt: skip
    t["grade_downgrade"] = (df["grade_champion"] - df["grade_origination_proxy"]) >= s[
        "sicr_grade_notches"
    ]
    t["pd_ratio"] = (df["pd_champion"] >= s["sicr_pd_ratio"] * df["pd_origination_proxy"]) & (
        df["pd_champion"] >= s["sicr_abs_pd_min"])  # fmt: skip
    stage = np.where(t["hc_90dpd"], 3, np.where(t[["hc_30dpd", "bureau_30dpd", "grade_downgrade",
                                                    "pd_ratio"]].any(axis=1), 2, 1))  # fmt: skip
    out = t.copy()
    out["stage"] = stage
    names = np.array(t.columns)
    out["triggers"] = [",".join(names[row]) for row in t.to_numpy()]
    return out

In [54]:
import acr.core.ecl.lifetime
show(acr.core.ecl.lifetime)

"""Lifetime PD term structures.

Both methods turn an account's 12-month PD into monthly hazards h_t (probability of default in
month t given survival to t-1); marginal default probability in month t is S_{t-1} * h_t.

1. Constant hazard (baseline): h_t = 1 - (1 - PD12) ** (1/12) for every month.
2. Markov: monthly transitions between DPD buckets estimated from bureau_balance (train
   customers only; decision D4). The chain gives the *shape* of default timing for a loan
   starting current. Each account's hazards are that shape scaled by a factor k chosen so the
   12-month cumulative PD equals the account's PD12. So both methods agree on year one and
   differ only in how risk is spread over the remaining life.
"""

from __future__ import annotations

import numpy as np
import pandas as pd


def constant_hazards(pd12: np.ndarray, horizon: int) -> np.ndarray:
    h = 1 - (1 - np.asarray(pd12, dtype="float64")) ** (1 / 12)
    return np.repeat(h[:, None], horizon, axis=1)


def marginal(hazards: np.ndarray) -> np.ndarray:
    surv = np.cumprod(1 - hazards, axis=1)
    prev = np.hstack([np.ones((hazards.shape[0], 1)), surv[:, :-1]])
    return prev * hazards


def transition_matrix(bb: pd.DataFrame, default_bucket: int) -> pd.DataFrame:
    """Monthly transition matrix over states 0..default_bucket-1 plus "D" (absorbing).

    ``bb`` has SK_ID_BUREAU, MONTHS_BALANCE and bucket (0-5, NaN for closed/unknown).
    Only consecutive observed months of the same credit count as a transition."""
    d = bb.dropna(subset=["bucket"]).sort_values(["SK_ID_BUREAU", "MONTHS_BALANCE"])
    state = np.minimum(d["bucket"].to_numpy(), default_bucket).astype(int)
    same = (d["SK_ID_BUREAU"].to_numpy()[1:] == d["SK_ID_BUREAU"].to_numpy()[:-1]) & (
        np.diff(d["MONTHS_BALANCE"].to_numpy()) == 1)  # fmt: skip
    k = default_bucket + 1
    counts = np.zeros((k, k))
    np.add.at(counts, (state[:-1][same], state[1:][same]), 1)
    counts[default_bucket, :] = 0
    counts[default_bucket, default_bucket] = 1  # default is absorbing
    unseen = counts.sum(axis=1) == 0
    counts[unseen, unseen] = 1  # a state never observed stays put rather than producing NaN
    P = counts / counts.sum(axis=1, keepdims=True)
    labels = [str(i) for i in range(default_bucket)] + ["D"]
    return pd.DataFrame(P, index=labels, columns=labels)


def markov_profile(P: pd.DataFrame, horizon: int) -> np.ndarray:
    """Monthly hazards of reaching default for a credit that starts current (state 0)."""
    M = P.to_numpy()
    v = np.zeros(M.shape[0])
    v[0] = 1.0
    cum = []
    for _ in range(horizon):
        v = v @ M
        cum.append(v[-1])
    cum = np.array(cum)
    prev = np.r_[0.0, cum[:-1]]
    return np.where(prev < 1, (cum - prev) / (1 - prev), 1.0)


def scaled_hazards(profile: np.ndarray, pd12: np.ndarray, iters: int = 60) -> np.ndarray:
    """Scale the Markov shape per account so that 1 - prod_{t<=12}(1 - k * lambda_t) = PD12."""
    lam12 = profile[:12]
    pd12 = np.asarray(pd12, dtype="float64")
    lo = np.zeros_like(pd12)
    hi = np.full_like(pd12, 1 / profile.max())
    for _ in range(iters):
        k = (lo + hi) / 2
        cum = 1 - np.prod(1 - k[:, None] * lam12[None, :], axis=1)
        too_low = cum < pd12
        lo, hi = np.where(too_low, k, lo), np.where(too_low, hi, k)
    k = (lo + hi) / 2
    return np.clip(k[:, None] * profile[None, :], 0, 1)

In [55]:
import acr.core.ecl.engine
show(acr.core.ecl.engine)

"""ECL computation per account, per scenario and lifetime method.

For performing loans (Stages 1 and 2):
    ECL = LGD * sum_{t=1..H} marginal_PD_t * EAD_t * DF_t,    DF_t = (1 + EIR) ** -t
with horizon H = min(12, term) for Stage 1 (12-month ECL) and H = term for Stage 2 (lifetime).
For Stage 3 (credit-impaired): PD = 1, ECL = LGD * EAD_0.

The engine works at LGD = 1 ("unit ECL"); ECL at any LGD is unit ECL * LGD, which makes the
LGD sensitivity exact. Forward-looking scenarios scale the 12-month PD before the term
structure is built; staging is held fixed across scenarios.
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np

from acr.core.ecl.lifetime import constant_hazards, marginal, scaled_hazards


@dataclass
class Book:
    stage: np.ndarray
    pd12: np.ndarray
    ead: np.ndarray  # (accounts, horizon) exposure at start of each month
    rate: np.ndarray  # monthly EIR
    term: np.ndarray  # months
    pd_floor: float

    @property
    def ead0(self) -> np.ndarray:
        return self.ead[:, 0]


def unit_ecl(book: Book, pd_scalar: float, method: str, profile: np.ndarray | None) -> dict:
    """ECL at LGD = 1 and lifetime PD, per account, for one scenario and method."""
    H = book.ead.shape[1]
    pd12 = np.clip(book.pd12 * pd_scalar, book.pd_floor, 0.9999)
    hz = (
        constant_hazards(pd12, H)
        if method == "constant_hazard"
        else scaled_hazards(profile[:H], pd12)
    )
    m = marginal(hz)
    t = np.arange(1, H + 1)[None, :]
    horizon = np.where(book.stage == 1, np.minimum(12, book.term), book.term)[:, None]
    m = np.where(t <= horizon, m, 0.0)
    df = (1 + book.rate[:, None]) ** -t
    unit = (m * book.ead * df).sum(axis=1)
    pd_h = m.sum(axis=1)
    s3 = book.stage == 3
    unit = np.where(s3, book.ead0, unit)
    pd_h = np.where(s3, 1.0, pd_h)
    return {"unit_ecl": unit, "pd_horizon": pd_h, "pd12_used": np.where(s3, 1.0, pd12)}


def floors_for(stage: np.ndarray, floors: dict) -> np.ndarray:
    """Stage-wise minimum provision rates (para 82(3)(x), unsecured retail). Every Stage 3
    account in this simulated book is in its first year of Stage 3."""
    return np.select([stage == 1, stage == 2, stage == 3],
                     [floors["stage1"], floors["stage2"], floors["stage3_0_1y"]])  # fmt: skip

In [56]:
import acr.nodes.ecl
show(acr.nodes.ecl)

"""Node 7: illustrative RBI ECL simulation on the pseudo-OOT book.

ILLUSTRATIVE ONLY. Home Credit is an application-level dataset, not a live portfolio. The OOT
partition is treated as a book at a simulated reporting date (seasoning in config). Regulatory
parameters come from config/ecl_rbi.yaml with paragraph citations to RBI/DOR/2026-27/398.

Data-learned inputs (implied rates, the DPD transition matrix) use train customers only.
Nothing here fits or changes the PD models: the ECL layer consumes their scored outputs.
"""

from __future__ import annotations

import numpy as np
import pandas as pd

from acr.audit import node
from acr.core.ecl import lifetime as L
from acr.core.ecl import rates as R
from acr.core.ecl.engine import Book, floors_for, unit_ecl
from acr.core.ecl.staging import INPUTS, allocate
from acr.core.io import read
from acr.core.legs.bureau import dpd_buckets
from acr.core.schema import ID
from acr.nodes._util import artefact_dir, cfg

METHODS = ("constant_hazard", "markov")


def _r(x, nd=6):
    if isinstance(x, dict):
        return {str(k): _r(v, nd) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_r(v, nd) for v in x]
    if isinstance(x, (float, np.floating)):
        return round(float(x), nd)
    if isinstance(x, np.integer):
        return int(x)
    return x


def _records(df: pd.DataFrame) -> list[dict]:
    return _r(df.reset_index().to_dict(orient="records"))


@node(
    "ecl",
    role="Illustrative RBI ECL on the OOT book: stage allocation with audited triggers, "
    "12-month and lifetime PD (constant hazard and bureau DPD Markov chain), amortising EAD, "
    "EIR discounting, weighted macro scenarios, LGD backstop and sensitivity, RBI stage floors.",
    reads=["config", "split", "dq", "selection", "model"],
    writes=["ecl"],
)
def ecl(view):
    c = cfg(view)
    e = c.ecl
    tables = {k: v["path"] for k, v in view["dq"]["tables"].items()}
    a = pd.read_parquet(view["split"]["assignment_path"]).set_index(ID)
    book_ids = a.index[a["partition"] == e.book.partition]
    train_ids = a.index[a["partition"] == "train"]

    # --- PDs and grades from the champion (consumed, never refitted) ---
    scored = pd.read_parquet(view["model"]["paths"]["scored"]).loc[book_ids]
    feats = pd.read_parquet(view["selection"]["paths"]["features_all"], columns=INPUTS).loc[
        book_ids
    ]
    df = scored.join(feats)

    # --- contract data at float64 precision ---
    app = read(tables["application"], [ID, "NAME_CONTRACT_TYPE", "AMT_CREDIT", "AMT_ANNUITY"],
               book_ids).set_index(ID).loc[book_ids]  # fmt: skip
    ctype = app["NAME_CONTRACT_TYPE"].astype("string")
    revolving = (ctype == "Revolving loans").to_numpy()

    # --- implied monthly rates from prior approved loans of TRAIN customers ---
    prev = read(tables["previous"], ["NAME_CONTRACT_TYPE", "NAME_CONTRACT_STATUS", "AMT_CREDIT",
                                     "AMT_ANNUITY", "CNT_PAYMENT"], train_ids)  # fmt: skip
    implied = R.implied_rates(prev)
    del prev
    rate = (
        ctype.map(lambda t: implied.get(t, e.term.fallback_monthly_rate)).astype(float).to_numpy()
    )

    # --- term and exposure schedule ---
    P = app["AMT_CREDIT"].to_numpy(dtype="float64")
    A = app["AMT_ANNUITY"].to_numpy(dtype="float64")
    term = R.term_months(P, np.nan_to_num(A, nan=1.0), rate, e.term.min_months, e.term.max_months)
    median_cash_term = int(np.median(term[~revolving & ~np.isnan(A)])) if (~revolving).any() else 12
    term = np.where(np.isnan(A), median_cash_term, term)
    term = np.where(revolving, e.revolving.life_months, term)
    s0 = e.book.seasoning_months
    H = int(max(term.max() - s0, 1))
    sched = R.balance_schedule(P, np.nan_to_num(A, nan=0.0), rate, term, H + s0)[:, s0:]
    rev_ead = np.where(
        np.arange(H)[None, :] < (term - s0)[:, None], (P * e.revolving.ccf)[:, None], 0.0
    )
    ead = np.where(revolving[:, None], rev_ead, sched)
    remaining = np.

In [57]:
from acr.nodes.ecl import ecl

step(ecl, state)
E = state["ecl"]
print("book:", E["book"])
print("implied monthly rates:", E["assumptions"]["implied_monthly_rates"])
print("stage counts:", E["stage_counts"], "| triggers:", E["stage_triggers"])
print("QC checks:", E["checks"])

ecl: wrote ['ecl']
book: {'partition': 'oot', 'accounts': 7688, 'ead_total': 4640397642.0, 'seasoning_months': 0, 'revolving_share': 0.09391259105098855}
implied monthly rates: {'Cash loans': 0.03333249111929963, 'Consumer loans': 0.033545351594563205}
stage counts: {1: 7283, 2: 399, 3: 6} | triggers: {'hc_90dpd': 6, 'hc_30dpd': 28, 'bureau_30dpd': 98, 'grade_downgrade': 89, 'pd_ratio': 286}
QC checks: {'scenario_weights_sum_to_1': True, 'ecl_non_negative': True, 'coverage_increases_with_stage': True, 'realised_default_rate_increases_with_stage': True, 'stage2_lifetime_pd_at_least_12m_pd': True}


In [58]:
# ECL by stage: model ECL, after RBI floors (final), coverage and realised default rate
display(pd.read_csv(E["paths"]["by_stage"], index_col=0).round(4))

,accounts,ead,mean_pd12,mean_pd_horizon,ecl_model,ecl_final,coverage_model,coverage_final,floor_binding_share,realised_default_rate
stage_1,7283.0,4.409464e+09,0.0783,0.0781,1.774046e+08,1.792082e+08,0.0402,0.0406,0.1443,0.0781
stage_2,399.0,2.269009e+08,0.1291,0.3681,3.034888e+07,3.084580e+07,0.1338,0.1359,0.1905,0.1278
stage_3,6.0,4.032603e+06,0.1437,1.0000,2.822822e+06,2.822822e+06,0.7000,0.7000,0.0000,0.1667
total,7688.0,4.640398e+09,0.0810,0.0939,2.105763e+08,2.128768e+08,0.0454,0.0459,0.1466,0.0808


In [59]:
# Scenarios (staging is fixed; PD scalars move ECL), lifetime-method comparison, LGD sensitivity
display(pd.read_csv(E["paths"]["by_scenario"], index_col=0).round(0))
display(pd.read_csv(E["paths"]["methods"], index_col=0).round(4))
display(pd.read_csv(E["paths"]["lgd"]).round(4))

,"base (w=0.60, PD x1.00)","upside (w=0.15, PD x0.85)","downside (w=0.25, PD x1.35)",probability_weighted
stage,,,,
1,166398990.0,141223821.0,225526463.0,177404583.0
2,28912635.0,25239359.0,36861565.0,30348876.0
3,2822822.0,2822822.0,2822822.0,2822822.0
total,198134448.0,169286002.0,265210850.0,210576281.0


,ecl_weighted,stage1_ecl,stage2_ecl,stage2_mean_lifetime_pd
constant_hazard,2.105763e+08,1.774046e+08,3.034888e+07,0.3681
markov,1.993036e+08,1.635816e+08,3.289921e+07,0.4238


,lgd,ecl_model,ecl_final,coverage_final
0,0.45,1.353705e+08,1.412427e+08,0.0304
1,0.55,1.654528e+08,1.694258e+08,0.0365
2,0.70,2.105763e+08,2.128768e+08,0.0459
3,0.80,2.406586e+08,2.422876e+08,0.0522


In [60]:
# Monthly DPD-bucket transition matrix estimated from bureau_balance (train customers)
display(pd.read_csv(E["paths"]["transition_matrix"], index_col=0).round(4))

,0,1,2,3,D
0,0.9747,0.0245,0.0007,0.0000,0.0001
1,0.5190,0.4129,0.0663,0.0018,0.0000
2,0.2416,0.1641,0.1848,0.3915,0.0181
3,0.1131,0.0536,0.0923,0.0833,0.6577
D,0.0000,0.0000,0.0000,0.0000,1.0000


## 11b. Gates and the rest of the graph

The routing functions in `graph.py` (shown in §3) read the status each gate node wrote and
pick the next node; each routing decision is logged to the audit trail. If a gate says
`halt`, the `halt` node writes `halt_report.md` listing the red findings.

Still stubs, built in Phase 6: `critic`, `run_extra_analyses`, `document` and
`grounding_check`. The critic loop already runs in dry-run mode
with canned responses, to prove the bounded loop and the whitelist work.

In [61]:
import acr.nodes.halt
show(acr.nodes.halt)

"""Halt node: reached when a gate returns red. Writes a clear halt report."""

from __future__ import annotations

import json
from pathlib import Path

from acr.audit import node
from acr.nodes.report import SUMMARY_KEYS, _llm_totals


@node(
    "halt",
    role="Stop the run after a red gate; write halt_report.md explaining why, and "
    "run_summary.json with everything computed before the halt.",
    reads=list(SUMMARY_KEYS),
    writes=["halted", "halt_reason", "report"],
)
def halt(view):
    red = [g for g in view.get("gate_log", []) if g["decision"] == "halt"]
    reason = "; ".join(f"{g['gate']}: {g['reason']}" for g in red) or "unspecified"
    lines = [
        "# Run halted",
        "",
        f"Run `{view['run_id']}` stopped at a red gate.",
        "",
        "| Gate | Decision | Reason |",
        "|---|---|---|",
    ]
    lines += [
        f"| {g['gate']} | {g['decision']} | {g['reason']} |" for g in view.get("gate_log", [])
    ]
    summary = {k: view.get(k) for k in SUMMARY_KEYS}
    summary.update(
        status="halted", halt_reason=reason, llm_totals=_llm_totals(view.get("llm_calls", []))
    )
    (Path(view["run_dir"]) / "run_summary.json").write_text(
        json.dumps(summary, indent=2, default=str)
    )
    path = Path(view["run_dir"]) / "halt_report.md"
    path.write_text("\n".join(lines) + "\n")
    return {
        "halted": True,
        "halt_reason": reason,
        "report": {"status": "halted", "halt_report": str(path)},
    }

In [62]:
# The audit trail written so far: one line per node start/end, gate and route
from acr.audit import read_events

events = pd.DataFrame(read_events(state["run_dir"]))
display(events[events["event"].isin(["node_end", "run_start"])][["ts", "event", "node", "seconds"]])

,ts,event,node,seconds
0,2026-10-19T06:01:28.060+00:00,run_start,NaN,NaN
2,2026-10-19T06:01:28.060+00:00,node_end,setup,0.001
4,2026-10-19T06:01:44.152+00:00,node_end,ingest_dq,16.027
6,2026-10-19T06:01:44.323+00:00,node_end,split,0.136
8,2026-10-19T06:01:44.601+00:00,node_end,leg_application,0.169
10,2026-10-19T06:01:45.191+00:00,node_end,leg_bureau,0.589
12,2026-10-19T06:01:46.089+00:00,node_end,leg_behavioural,0.898
14,2026-10-19T06:01:46.319+00:00,node_end,leg_previous,0.230
16,2026-10-19T06:01:50.671+00:00,node_end,select,4.268
18,2026-10-19T06:01:57.330+00:00,node_end,model,6.554


## 12. Offline tuning (`acr tune`): not part of the graph

Hyperparameters were tuned **offline**, by deterministic 3-fold cross-validation **inside the
train partition only**. Test and OOT are never read. The winner is the **simplest
configuration within one standard error** of the best CV Gini. The tuner writes a report;
the winners were reviewed and then frozen into `base.yaml` by hand (decision D22).

Not re-run here (it takes about 35 minutes on the full data). The code:

In [63]:
import acr.tuning
show(acr.tuning)

"""Offline, deterministic hyperparameter tuning (``acr tune``). Not part of the graph.

Everything here is cross-validation *inside the train partition* of an existing run. The
test and OOT partitions are never read, so tuning cannot leak holdout information. Results
go to a report; a human reviews them and freezes the winners into ``config/base.yaml``.
The pipeline then runs on fixed config, as before.

Winner selection uses the one-standard-error rule: among configurations whose mean CV Gini
is within one standard error of the best, pick the simplest.
"""

from __future__ import annotations

import itertools
import json
import pickle
import time
from dataclasses import dataclass
from datetime import UTC, datetime
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
import yaml
from pydantic import BaseModel, ConfigDict, field_validator
from sklearn.model_selection import StratifiedKFold

from acr.config import (
    CONFIG_DIR,
    MAX_SCORECARD_FEATURES,
    AppConfig,
    check_depth,
    resolve_path,
)
from acr.core import binning as B
from acr.core import challenger as C
from acr.core.guards import require_partition
from acr.core.metrics import gini
from acr.core.scorecard import fit_scorecard
from acr.core.selection import prune, select_features


class ScorecardGrid(BaseModel):
    model_config = ConfigDict(extra="forbid")
    max_n_bins: list[int]
    min_bin_size: list[float]
    max_features: list[int]
    iv_min: list[float]

    @field_validator("max_features")
    @classmethod
    def _cap(cls, v: list[int]) -> list[int]:
        if max(v) > MAX_SCORECARD_FEATURES:
            raise ValueError(
                f"max_features grid exceeds the {MAX_SCORECARD_FEATURES}-variable limit"
            )
        return v


class LGBMGrid(BaseModel):
    model_config = ConfigDict(extra="forbid")
    max_depth: list[int]

    @field_validator("max_depth")
    @classmethod
    def _depth(cls, v: list[int]) -> list[int]:
        return [check_depth(d) for d in v]

    num_leaves: list[int]
    min_child_samples: list[int]
    colsample_bytree: list[float]
    reg_lambda: list[float]


class TuningCfg(BaseModel):
    model_config = ConfigDict(extra="forbid")
    folds: int
    selection_rule: str
    scorecard: ScorecardGrid
    lightgbm: LGBMGrid


def load_tuning(config_dir: Path = CONFIG_DIR) -> TuningCfg:
    return TuningCfg.model_validate(yaml.safe_load((config_dir / "tuning.yaml").read_text()))


@dataclass
class TrainData:
    features: pd.DataFrame  # train rows only
    y: pd.Series
    dictionary: pd.DataFrame
    candidates: list[str]  # candidates surviving the missingness screen
    categorical: list[str]
    source_run: str


def load_train(run_dir: Path) -> TrainData:
    sel = run_dir / "artefacts" / "selection"
    meta = json.loads((sel / "selection.json").read_text())
    with open(sel / "binning_process.pkl", "rb") as f:
        live = list(pickle.load(f).variable_names)
    a = pd.read_parquet(run_dir / "artefacts" / "split" / "assignment.parquet")
    feats = pd.read_parquet(sel / "features_all.parquet")
    train_ids = a.loc[a["partition"] == "train", "SK_ID_CURR"]
    Xtr = require_partition(feats.loc[train_ids], a, "train")
    y = a.set_index("SK_ID_CURR").loc[train_ids, "TARGET"]
    return TrainData(Xtr, y, pd.read_csv(sel / "feature_dictionary.csv"), live,
                     [c for c in live if c in meta["categorical"]], run_dir.name)  # fmt: skip


def one_se(results: pd.DataFrame, simplicity: list[str], ascending: list[bool]) -> pd.Series:
    best = results.loc[results["cv_gini_mean"].idxmax()]
    eligible = results[results["cv_gini_mean"] >= best["cv_gini_mean"] - best["cv_gini_se"]]
    return eligible.sort_values(simplicity, ascending=ascending).iloc[0]


def _folds(y: pd.Series, k: int, seed: int):
    return list(StratifiedKFold(k, shuffle=True, random_state=seed).split(np.zeros(len(y)), y))


def _summarise(rows: list[dict], keys: list[str]) -> pd.Da

## 13. Tests

The test suite runs on a **synthetic** Home Credit-shaped dataset (`tests/synthetic.py`),
never the Kaggle files. What each file guards:

| File | Guards |
|---|---|
| `test_split.py` | 56.25/18.75/25 proportions, stratification, no ID overlap, determinism, sampling |
| `test_dq.py` | DQ checks and the DQ halt gate |
| `test_io.py` | downcasting preserves values; cache reuse and rebuild |
| `test_legs.py` | leg shape and dictionaries; legs never touch TARGET; hand-checked DPD maths |
| `test_modelling.py` | **leakage**: shuffling test/OOT targets changes nothing learned; reproducibility; points table reproduces scores |
| `test_tuning.py` | tuning ignores holdout targets, is deterministic, respects the 15-variable and depth-10 limits |
| `test_validation.py` | gate logic; a tampered artefact halts the run; the validator imports no fitting code |
| `test_contract.py` | the node read/write contract is enforced |
| `test_gates.py` | halt routing; critic loop bounded at 2; off-menu critic requests rejected; LLM nodes skip cleanly |
| `test_graph.py`, `test_config.py` | graph wiring and fan-out/join; config validation and limits |

Run them with `.venv/bin/pytest`.